In [1]:
# Cell 1 - Environment Setup, Deterministic Seeds, TF Memory Growth, Log Silencing, and Purge VRAM Helper
import os

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"         
os.environ["PYTHONHASHSEED"] = "42"
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

import warnings
warnings.filterwarnings("ignore", message="pkg_resources is deprecated")

import gc
import random
import logging
import numpy as np
import torch
import tensorflow as tf

logging.getLogger("absl").setLevel(logging.ERROR)
tf.get_logger().setLevel("ERROR")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
tf.random.set_seed(SEED)

DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
NUM_WORKERS = min(4, os.cpu_count() or 2)

tf_memory_growth = False
gpus = tf.config.list_physical_devices("GPU")
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        tf_memory_growth = True
    except RuntimeError as e:
        print(f"tf_memory_growth_error: {e}")

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True, warn_only=True)  # warn_only: tidak crash di op non-deterministik
    gpu_name = torch.cuda.get_device_name(0)
    free_vram, total_vram = torch.cuda.mem_get_info(0)
    print(f"hardware_target | device={DEVICE} | gpu={gpu_name} | vram_free={free_vram / 1024**3:.2f}GB | total_vram={total_vram / 1024**3:.2f}GB | tf_memory_growth={tf_memory_growth}")
else:
    print(f"hardware_target | device={DEVICE} (CPU mode) | tf_memory_growth={tf_memory_growth}")

# Helper DataLoader (dipakai di cell berikutnya):
# DataLoader(..., num_workers=NUM_WORKERS, worker_init_fn=seed_worker, generator=g)
def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)

g = torch.Generator()
g.manual_seed(SEED)

def purge_vram():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.synchronize()
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

print(f"environment_ready | seed={SEED} | workers={NUM_WORKERS}")

hardware_target | device=cuda:0 | gpu=Tesla T4 | vram_free=14.46GB | total_vram=14.56GB | tf_memory_growth=True
environment_ready | seed=42 | workers=4


In [2]:
# Cell 2 - Remote Dataset Discovery & Self-Contained Combinatorial 4-Fold Generator
from itertools import product
from pathlib import Path
import numpy as np
import pandas as pd

candidate_base_dirs = [
    Path("/kaggle/input/competitions/schematics-2026"),
    Path("/kaggle/input/schematics-2026"),
    Path("schematics-2026"),
    Path("C:/Github/schematics-npc-dsc-2026/schematics-2026"),
]

base_dir = next((p for p in candidate_base_dirs if p.exists()), None)
if base_dir is None:
    raise FileNotFoundError("Direktori dataset schematics-2026 tidak ditemukan.")

WORK_DIR = Path("/kaggle/working") if str(base_dir).startswith("/kaggle") else Path("schematics_day4_all_artifacts")
WORK_DIR.mkdir(parents=True, exist_ok=True)

train_meta_path = base_dir / "train" / "metadata.csv"
sub_path = base_dir / "sample_submission.csv"

train_df = pd.read_csv(train_meta_path)
sub_df = pd.read_csv(sub_path)

train_df["label"] = train_df["label"].str.strip()
train_audio_dir = base_dir / "train" / "audio"
test_audio_dir = base_dir / "test" / "audio"

train_df["filepath"] = [str(train_audio_dir / f"{fid}.flac") for fid in train_df["file_id"]]
sub_df["filepath"] = [str(test_audio_dir / f"{fid}.flac") for fid in sub_df["file_id"]]

assert len(train_df) == 1826 and len(sub_df) == 814, "Jumlah baris data tidak sesuai"
assert train_df["file_id"].nunique() == 1826 and sub_df["file_id"].nunique() == 814, "file_id duplikat"
assert train_df["source_group"].notna().all(), "Ada source_group yang kosong (NaN)"
assert (train_df.groupby("source_group")["label"].nunique() == 1).all(), "Ada source_group dengan lebih dari satu label"

missing_train = [p for p in train_df["filepath"] if not Path(p).exists()]
missing_test = [p for p in sub_df["filepath"] if not Path(p).exists()]
assert not missing_train and not missing_test, (
    f"File audio hilang | train={len(missing_train)} (contoh: {missing_train[:2]}) | "
    f"test={len(missing_test)} (contoh: {missing_test[:2]})"
)

folds_meta_path = WORK_DIR / "metadata_with_folds.csv"
N_FOLDS = 4

if folds_meta_path.exists():
    saved_folds = pd.read_csv(folds_meta_path)
    if {"file_id", "fold"}.issubset(saved_folds.columns) and len(saved_folds) == len(train_df):
        fold_map = saved_folds.set_index("file_id")["fold"]
        if set(fold_map.index) == set(train_df["file_id"]):
            train_df["fold"] = train_df["file_id"].map(fold_map).astype(int)

if "fold" not in train_df.columns:
    grp_meta = (
        train_df.groupby(["label", "source_group"])
        .size()
        .reset_index(name="n_samples")
        .sort_values(["label", "n_samples", "source_group"], ascending=[True, False, True])
    )
    cend_grps = grp_meta[grp_meta["label"] == "Cendrawasih"][["source_group", "n_samples"]].values.tolist()
    kuti_grps = grp_meta[grp_meta["label"] == "Kutilang"][["source_group", "n_samples"]].values.tolist()
    bond_grps = grp_meta[grp_meta["label"] == "Bondol Jawa"][["source_group", "n_samples"]].values.tolist()

    assert len(cend_grps) == N_FOLDS and len(kuti_grps) == N_FOLDS and len(bond_grps) == 7, (
        f"Jumlah grup tidak sesuai | cend={len(cend_grps)} kuti={len(kuti_grps)} bond={len(bond_grps)}"
    )

    group_to_fold = {}
    for f in range(N_FOLDS):
        group_to_fold[cend_grps[f][0]] = f
        group_to_fold[kuti_grps[N_FOLDS - 1 - f][0]] = f

    base_tot = [cend_grps[f][1] + kuti_grps[N_FOLDS - 1 - f][1] for f in range(N_FOLDS)]

    best_score = float("inf")
    best_bond_assign = None
    for assign in product(range(N_FOLDS), repeat=len(bond_grps)):
        if len(set(assign)) < N_FOLDS:
            continue
        fold_bond = [0] * N_FOLDS
        fold_tot = list(base_tot)
        for (_, sz), f in zip(bond_grps, assign):
            fold_bond[f] += sz
            fold_tot[f] += sz
        score = np.std(fold_bond) + 0.5 * np.std(fold_tot)
        if score < best_score:
            best_score = score
            best_bond_assign = assign

    for (grp_id, _), f in zip(bond_grps, best_bond_assign):
        group_to_fold[grp_id] = f

    train_df["fold"] = train_df["source_group"].map(group_to_fold)
    assert train_df["fold"].notna().all(), "Ada source_group yang tidak ter-assign ke fold"
    train_df["fold"] = train_df["fold"].astype(int)
    train_df.to_csv(folds_meta_path, index=False)

val_groups = [set(train_df[train_df["fold"] == f]["source_group"].unique()) for f in range(N_FOLDS)]
leakage = sum(len(val_groups[i] & val_groups[j]) for i in range(N_FOLDS) for j in range(i + 1, N_FOLDS))
assert leakage == 0, f"Group leakage terdeteksi: {leakage}"

fold_cendra = [int((train_df[train_df["fold"] == f]["label"] == "Cendrawasih").sum()) for f in range(N_FOLDS)]
assert fold_cendra == [35, 29, 16, 3], f"Distribusi Cendrawasih menyimpang: {fold_cendra}"

print(f"metadata_locked | train={len(train_df)} | test={len(sub_df)} | fold_cendra={fold_cendra} | group_leakage={leakage}")

metadata_locked | train=1826 | test=814 | fold_cendra=[35, 29, 16, 3] | group_leakage=0


In [3]:
# Cell 3 - Fast Dual-Stream DSP RAM Pre-Caching (32 kHz 5s & 48 kHz 3s)
from math import gcd
import time
import numpy as np
import soundfile as sf
from scipy.signal import resample_poly, butter, sosfilt
from joblib import Parallel, delayed

t0_dsp = time.time()

# 1. Polyphase Resampler (48kHz -> 32kHz)
def resample_poly_32k(wav: np.ndarray, orig_sr: int = 48000, target_sr: int = 32000) -> np.ndarray:
    if orig_sr == target_sr:
        return wav.astype(np.float32)
    g = gcd(orig_sr, target_sr)
    up, down = target_sr // g, orig_sr // g
    return resample_poly(wav, up, down).astype(np.float32)

# 2. Butterworth High-Pass 100 Hz
def butter_highpass_100hz(wav: np.ndarray, sr: int = 32000, cutoff: float = 100.0, order: int = 4) -> np.ndarray:
    sos = butter(order, cutoff, btype="highpass", fs=sr, output="sos")
    return sosfilt(sos, wav).astype(np.float32)

# 3. Linear Peak Normalization
def linear_peak_norm(wav: np.ndarray, target_peak: float = 0.95) -> np.ndarray:
    peak = float(np.max(np.abs(wav)))
    if peak > 1e-6:
        return (wav / peak * target_peak).astype(np.float32)
    return wav.astype(np.float32)

# 4. Inline VAD Window Indices (500-8000 Hz) - O(N) via cumulative sum
def _compute_vad_window_indices(
    wav: np.ndarray,
    sr: int = 32000,
    duration_sec: float = 10.0,
    band: tuple = (500, 8000),
) -> tuple:
    target_len = int(sr * duration_sec)
    cur_len = len(wav)
    if cur_len <= target_len:
        return 0, target_len
    sos_bp = butter(2, [band[0], band[1]], btype="bandpass", fs=sr, output="sos")
    bp = sosfilt(sos_bp, wav).astype(np.float64)
    stride = int(sr * 0.25)
    n_steps = max(1, (cur_len - target_len) // stride + 1)
    cs = np.concatenate(([0.0], np.cumsum(bp ** 2)))
    starts = np.arange(n_steps) * stride
    energies = cs[starts + target_len] - cs[starts]
    best_start = int(starts[int(np.argmax(energies))])
    return best_start, best_start + target_len

# 5. Dynamic Equidistant 3-Window Extractor (Jalur Perch: 32 kHz, 5.0s = 160,000 sampel)
def extract_perch_3windows_5s(wav: np.ndarray, sr: int = 32000) -> np.ndarray:
    win_len = int(sr * 5.0)       # 160,000
    target_10s = int(sr * 10.0)   # 320,000
    cur_len = len(wav)

    if cur_len < win_len:
        pad_len = win_len - cur_len
        w_pad = np.pad(wav, (0, pad_len), mode="reflect" if pad_len < cur_len else "constant")
        w_pad = linear_peak_norm(w_pad, target_peak=0.95)
        return np.stack([w_pad, w_pad, w_pad], axis=0).astype(np.float32)
    elif cur_len < target_10s:
        t0 = 0
        t2 = cur_len - win_len
        t1 = (t0 + t2) // 2
        w0 = linear_peak_norm(wav[t0 : t0 + win_len], target_peak=0.95)
        w1 = linear_peak_norm(wav[t1 : t1 + win_len], target_peak=0.95)
        w2 = linear_peak_norm(wav[t2 : t2 + win_len], target_peak=0.95)
        return np.stack([w0, w1, w2], axis=0).astype(np.float32)
    else:
        t_start, t_end = _compute_vad_window_indices(wav, sr=sr, duration_sec=10.0, band=(500, 8000))
        chunk10 = wav[t_start:t_end]
        stride = int(sr * 2.5)   # 80,000
        w0 = linear_peak_norm(chunk10[0 : win_len], target_peak=0.95)
        w1 = linear_peak_norm(chunk10[stride : stride + win_len], target_peak=0.95)
        w2 = linear_peak_norm(chunk10[2 * stride : 2 * stride + win_len], target_peak=0.95)
        return np.stack([w0, w1, w2], axis=0).astype(np.float32)

# 6. Dynamic Equidistant 3-Window Extractor (Jalur BirdNET: 48 kHz, 3.0s = 144,000 sampel)
def extract_birdnet_3windows_3s(wav: np.ndarray, sr: int = 48000) -> np.ndarray:
    chunk_len = int(sr * 3.0)  # 144,000
    cur_len = len(wav)
    if cur_len < chunk_len:
        pad_len = chunk_len - cur_len
        wav = np.pad(wav, (0, pad_len), mode="wrap")
        cur_len = len(wav)
    w1 = wav[:chunk_len]
    mid = (cur_len - chunk_len) // 2
    w2 = wav[mid : mid + chunk_len]
    w3 = wav[-chunk_len:]
    return np.stack([w1, w2, w3], axis=0).astype(np.float32)

# 7. Dual-Stream Process Single Audio (Single Disk Read) + metadata untuk QC
def process_single_audio_dual(filepath: str):
    data, orig_sr = sf.read(str(filepath), dtype="float32")
    mono = data.mean(axis=1) if data.ndim > 1 else data
    dur_sec = len(mono) / orig_sr

    # Jalur Perch 32k
    wav_32k = resample_poly_32k(mono, orig_sr=orig_sr, target_sr=32000)
    wav_hp_32k = butter_highpass_100hz(wav_32k, sr=32000, cutoff=100.0)
    wins_32k = extract_perch_3windows_5s(wav_hp_32k, sr=32000)

    # Jalur BirdNET 48k
    if orig_sr == 48000:
        wav_48k = mono
    else:
        g = gcd(orig_sr, 48000)
        wav_48k = resample_poly(mono, 48000 // g, orig_sr // g).astype(np.float32)
    wins_48k = extract_birdnet_3windows_3s(wav_48k, sr=48000)

    n_silent = int((np.abs(wins_32k).max(axis=1) < 1e-4).sum())
    return wins_32k, wins_48k, dur_sec, n_silent

# 8. Cache satu split langsung ke array pre-allocated (tanpa salinan ganda)
def cache_split(filepaths, name):
    n = len(filepaths)
    arr_32k = np.empty((n, 3, 160000), dtype=np.float32)
    arr_48k = np.empty((n, 3, 144000), dtype=np.float32)
    durs = np.empty(n, dtype=np.float32)
    silent = np.zeros(n, dtype=np.int16)

    gen = Parallel(n_jobs=NUM_WORKERS, backend="loky", return_as="generator")(
        delayed(process_single_audio_dual)(fp) for fp in filepaths
    )  # generator tetap berurutan; butuh joblib >= 1.3
    for i, (w32, w48, d, s) in enumerate(gen):
        arr_32k[i] = w32
        arr_48k[i] = w48
        durs[i] = d
        silent[i] = s
    print(f"{name}_cached | n={n}")
    return arr_32k, arr_48k, durs, silent

print(f"pre_caching_dual_audio_to_ram (workers={NUM_WORKERS})...")

train_32k, train_48k, train_durs, train_silent = cache_split(train_df["filepath"].tolist(), "train")
test_32k, test_48k, test_durs, test_silent = cache_split(sub_df["filepath"].tolist(), "test")

n_tr, n_te = len(train_df), len(sub_df)
assert train_32k.shape == (n_tr, 3, 160000) and test_32k.shape == (n_te, 3, 160000)
assert train_48k.shape == (n_tr, 3, 144000) and test_48k.shape == (n_te, 3, 144000)
for arr_name, arr in [("train_32k", train_32k), ("test_32k", test_32k),
                      ("train_48k", train_48k), ("test_48k", test_48k)]:
    assert np.isfinite(arr).all(), f"NaN/Inf terdeteksi di {arr_name}"

all_durs = np.concatenate([train_durs, test_durs])
short_5s = int((all_durs < 5.0).sum())   # Perch: di-pad
short_3s = int((all_durs < 3.0).sum())   # BirdNET: di-wrap
silent_files = int((train_silent > 0).sum() + (test_silent > 0).sum())

elapsed_c3 = time.time() - t0_dsp
print(
    f"ram_caching_complete | train={len(train_32k)} | test={len(test_32k)} | "
    f"padded_perch(<5s)={short_5s} | wrapped_birdnet(<3s)={short_3s} | "
    f"silent_files={silent_files} | dur_min={all_durs.min():.2f}s | elapsed={elapsed_c3:.1f}s"
)

pre_caching_dual_audio_to_ram (workers=4)...
train_cached | n=1826
test_cached | n=814
ram_caching_complete | train=1826 | test=814 | padded_perch(<5s)=0 | wrapped_birdnet(<3s)=0 | silent_files=0 | dur_min=5.27s | elapsed=211.1s


In [4]:
# Cell 4 - Batch Extraction Google Perch v2 (Embedding 1536-d + Candidate Logits 14-d)
import contextlib
import json
import os
import time
from pathlib import Path
import numpy as np
import pandas as pd
import tensorflow as tf
import tensorflow_hub as hub

t0_perch = time.time()

# Peredam stderr level file descriptor (log native XLA/CUDA). Traceback Python tetap tampil.
@contextlib.contextmanager
def silence_native_stderr():
    saved = os.dup(2)
    devnull = os.open(os.devnull, os.O_WRONLY)
    try:
        os.dup2(devnull, 2)
        yield
    finally:
        os.dup2(saved, 2)
        os.close(saved)
        os.close(devnull)

# 1. Deteksi dan Muat Model Google Perch v2
perch_candidates = [
    Path("/kaggle/input/bird-vocalization-classifier/tensorFlow2/perch_v2/2"),
    Path("/kaggle/input/models/google/bird-vocalization-classifier/tensorFlow2/perch_v2/2"),
    Path("/kaggle/input/google-perch-v2"),
    Path("/kaggle/input/perch-v2"),
    WORK_DIR / "perch_v2",
    Path("perch_v2"),
]

perch_model = None
resolved_model_dir = None
for p_dir in perch_candidates:
    if p_dir.exists():
        try:
            perch_model = hub.load(str(p_dir))
            resolved_model_dir = p_dir.resolve()
            print(f"perch_model_loaded | type=local | path={resolved_model_dir}")
            break
        except Exception as e:
            print(f"  [warn] Gagal memuat dari {p_dir}: {e}")

if perch_model is None:
    tf_url = "https://www.kaggle.com/models/google/bird-vocalization-classifier/tensorFlow2/perch_v2/2"
    print(f"loading_perch_from_tfhub: {tf_url}...")
    try:
        resolved_path_str = hub.resolve(tf_url)
        resolved_model_dir = Path(resolved_path_str)
        perch_model = hub.load(resolved_path_str)
    except Exception as e:
        raise RuntimeError(
            "Model Perch v2 tidak ditemukan secara lokal dan gagal diunduh. "
            "Tambahkan model lewat 'Add Input' di Kaggle atau aktifkan Internet. "
            f"Detail: {e}"
        )
    print(f"perch_model_loaded | type=tfhub | cache={resolved_model_dir}")

infer_fn = perch_model.signatures["serving_default"]

# 2. Verifikasi Output Signature & Taksa Model (14.795)
with silence_native_stderr():
    dummy_out = infer_fn(inputs=tf.constant(np.zeros((1, 160000), dtype=np.float32)))
logit_key = next((k for k in ["label", "logits", "output_0", "predictions"] if k in dummy_out), None)
if logit_key is None:
    logit_key = [k for k in dummy_out.keys() if k not in ("embedding", "spatial_embedding", "spectrogram")][0]

emb_dim = int(dummy_out["embedding"].shape[-1])
num_taxa = int(dummy_out[logit_key].shape[-1])
print(f"perch_signature_verified | emb_dim={emb_dim} | num_taxa={num_taxa} | logit_key='{logit_key}'")
assert emb_dim == 1536 and num_taxa == 14795, f"Dimensi model tidak sesuai: emb={emb_dim}, taxa={num_taxa}"

# 3. Resolusi labels.csv (dibatasi ke folder model & WORK_DIR)
label_candidates = []
if resolved_model_dir and resolved_model_dir.exists():
    label_candidates.extend(sorted(resolved_model_dir.rglob("*.csv")))
if WORK_DIR.exists():
    label_candidates.extend(sorted(WORK_DIR.rglob("*label*.csv")))
for p_in in perch_candidates:
    if p_in.exists() and (resolved_model_dir is None or p_in.resolve() != resolved_model_dir):
        label_candidates.extend(sorted(p_in.rglob("*.csv")))

labels_df = None
for p_lbl in label_candidates:
    try:
        df_c = pd.read_csv(p_lbl)
        if len(df_c) == num_taxa:
            labels_df = df_c
            break
        df_c_nohead = pd.read_csv(p_lbl, header=None)
        if len(df_c_nohead) == num_taxa:
            labels_df = df_c_nohead
            labels_df.columns = [f"col_{i}" for i in range(labels_df.shape[1])]
            break
    except Exception:
        continue

assert labels_df is not None and len(labels_df) == num_taxa, "labels.csv (14.795 baris) tidak ditemukan"

# 4. Indeks Taksa Target (hanya nama ilmiah, tanpa substring genus longgar)
BONDOL_SCI = "Lonchura leucogastroides"
KUTILANG_SCI = "Pycnonotus aurigaster"
CENDRA_SCI = [
    "Cicinnurus regius", "Paradisaea apoda", "Paradisaea decora", "Paradisaea guilielmi",
    "Paradisaea minor", "Paradisaea raggiana", "Paradisaea rubra", "Seleucidis melanoleucus",
]

cols_to_check = [c for c in labels_df.columns if not pd.api.types.is_numeric_dtype(labels_df[c])]
assert cols_to_check, "Tidak ada kolom teks di labels.csv"

def find_taxa(sci_name):
    """Cocok jika sel (setelah strip) sama persis dengan nama ilmiah, tanpa regex."""
    matched = set()
    target = sci_name.strip().lower()
    for c in cols_to_check:
        mask = labels_df[c].astype(str).str.strip().str.lower() == target
        matched.update(labels_df.index[mask].tolist())
    return sorted(i for i in matched if 0 <= i < num_taxa)

def label_text(i):
    return " | ".join(str(v) for v in labels_df.iloc[i].tolist())

def resolve_single(sci_name):
    hits = find_taxa(sci_name)
    assert len(hits) == 1, f"'{sci_name}': diharapkan tepat 1 taksa, diperoleh {len(hits)}: {[label_text(i) for i in hits]}"
    return hits[0]

idx_bondol = resolve_single(BONDOL_SCI)
idx_kutilang = resolve_single(KUTILANG_SCI)
cendra_candidates = [resolve_single(sp) for sp in CENDRA_SCI]   # urutan sesuai CENDRA_SCI
assert len(set(cendra_candidates + [idx_bondol, idx_kutilang])) == 2 + len(CENDRA_SCI), "Ada indeks taksa duplikat"

print(f"  Bondol    -> [{idx_bondol}] {label_text(idx_bondol)}")
print(f"  Kutilang  -> [{idx_kutilang}] {label_text(idx_kutilang)}")
for i in cendra_candidates:
    print(f"  Cendra    -> [{i}] {label_text(i)}")

all_target_cands = [idx_bondol, idx_kutilang] + list(cendra_candidates)
non_target_mask = np.ones(num_taxa, dtype=bool)
non_target_mask[all_target_cands] = False
n_cand_cendra = len(cendra_candidates)
n_summary_dim = 1 + 1 + n_cand_cendra + 4  # 14 dimensi

print(f"taxa_indices_resolved | Bondol={idx_bondol} | Kutilang={idx_kutilang} | Cendra_cands={n_cand_cendra} | summary_dim={n_summary_dim}")
assert n_cand_cendra == 8 and n_summary_dim == 14, (
    f"Ekspektasi 8 kandidat Cendrawasih dan 14 dimensi, diperoleh {n_cand_cendra} dan {n_summary_dim}"
)

with open(WORK_DIR / "perch_taxa_indices.json", "w") as f:
    json.dump({
        "bondol": int(idx_bondol),
        "kutilang": int(idx_kutilang),
        "cendra": {sp: int(i) for sp, i in zip(CENDRA_SCI, cendra_candidates)},
        "summary_layout": ["bondol", "kutilang"] + CENDRA_SCI + ["max_nt", "top5_nt", "energy", "entropy"],
        "summary_dim": int(n_summary_dim),
    }, f, indent=2)

# 5. Ekstraksi Batch Langsung dari RAM Array (32kHz)
def extract_perch_features_from_ram(arr_32k: np.ndarray, batch_size: int = 32):
    n_files, n_windows, win_len = arr_32k.shape
    total_windows = n_files * n_windows
    flat_wavs = arr_32k.reshape(total_windows, win_len)

    embs_out = np.empty((total_windows, emb_dim), dtype=np.float32)
    summs_out = np.empty((total_windows, n_summary_dim), dtype=np.float32)

    for i in range(0, total_windows, batch_size):
        batch = flat_wavs[i : i + batch_size]
        res = infer_fn(inputs=tf.constant(batch))

        e_b = res["embedding"].numpy()
        l_b = res[logit_key].numpy()
        if e_b.ndim > 2:
            e_b = np.mean(e_b, axis=1)
        if l_b.ndim > 2:
            l_b = np.mean(l_b, axis=1)

        # L2-normalisasi embedding
        e_b = e_b / (np.linalg.norm(e_b, axis=-1, keepdims=True) + 1e-12)

        # Ringkasan 14-d logit
        z_b = l_b[:, idx_bondol, None]
        z_k = l_b[:, idx_kutilang, None]
        z_c = l_b[:, cendra_candidates]
        nt_logits = l_b[:, non_target_mask]
        max_nt = np.max(nt_logits, axis=1, keepdims=True)
        top5_nt = np.mean(np.partition(nt_logits, -5, axis=1)[:, -5:], axis=1, keepdims=True)

        max_l = np.max(l_b, axis=1, keepdims=True)
        exp_l = np.exp(np.clip(l_b - max_l, -50.0, 50.0))
        sum_exp = np.sum(exp_l, axis=1, keepdims=True)
        probs = exp_l / (sum_exp + 1e-12)
        entropy = -np.sum(probs * np.log(probs + 1e-12), axis=1, keepdims=True)
        energy = max_l + np.log(sum_exp + 1e-12)

        s_b = np.hstack([z_b, z_k, z_c, max_nt, top5_nt, energy, entropy]).astype(np.float32)

        embs_out[i : i + len(batch)] = e_b
        summs_out[i : i + len(batch)] = s_b

    embs_3w = embs_out.reshape(n_files, n_windows, emb_dim)
    summs_3w = summs_out.reshape(n_files, n_windows, n_summary_dim)
    return embs_3w, summs_3w

print("extracting_perch_features_train...")
with silence_native_stderr():
    train_embs_3w, train_cand_logits_3w = extract_perch_features_from_ram(train_32k, batch_size=32)

print("extracting_perch_features_test...")
with silence_native_stderr():
    test_embs_3w, test_cand_logits_3w = extract_perch_features_from_ram(test_32k, batch_size=32)

# 6. Validasi dulu, baru simpan
n_tr, n_te = len(train_df), len(sub_df)
assert train_embs_3w.shape == (n_tr, 3, 1536) and test_embs_3w.shape == (n_te, 3, 1536)
assert train_cand_logits_3w.shape == (n_tr, 3, n_summary_dim) and test_cand_logits_3w.shape == (n_te, 3, n_summary_dim)
for arr_name, arr in [("train_embs", train_embs_3w), ("test_embs", test_embs_3w),
                      ("train_cand", train_cand_logits_3w), ("test_cand", test_cand_logits_3w)]:
    assert np.isfinite(arr).all(), f"NaN/Inf terdeteksi di {arr_name}"

np.save(WORK_DIR / "perch_v2_train_emb.npy", train_embs_3w)
np.save(WORK_DIR / "train_raw_cand_summary.npy", train_cand_logits_3w)
np.save(WORK_DIR / "perch_v2_test_emb.npy", test_embs_3w)
np.save(WORK_DIR / "test_raw_cand_summary.npy", test_cand_logits_3w)

elapsed_c4 = time.time() - t0_perch
print(
    f"perch_extracted | train_emb={train_embs_3w.shape} | test_emb={test_embs_3w.shape} | "
    f"candidate_logits={train_cand_logits_3w.shape[-1]} | elapsed={elapsed_c4:.1f}s"
)

loading_perch_from_tfhub: https://www.kaggle.com/models/google/bird-vocalization-classifier/tensorFlow2/perch_v2/2...


I0000 00:00:1792307586.945743      48 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1792307586.950564      48 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


perch_model_loaded | type=tfhub | cache=/kaggle/input/models/google/bird-vocalization-classifier/tensorflow2/perch_v2/2
perch_signature_verified | emb_dim=1536 | num_taxa=14795 | logit_key='label'
  Bondol    -> [7329] Lonchura leucogastroides
  Kutilang  -> [11701] Pycnonotus aurigaster
  Cendra    -> [2893] Cicinnurus regius
  Cendra    -> [9634] Paradisaea apoda
  Cendra    -> [9635] Paradisaea decora
  Cendra    -> [9636] Paradisaea guilielmi
  Cendra    -> [9637] Paradisaea minor
  Cendra    -> [9638] Paradisaea raggiana
  Cendra    -> [9639] Paradisaea rubra
  Cendra    -> [12538] Seleucidis melanoleucus
taxa_indices_resolved | Bondol=7329 | Kutilang=11701 | Cendra_cands=8 | summary_dim=14
extracting_perch_features_train...
extracting_perch_features_test...
perch_extracted | train_emb=(1826, 3, 1536) | test_emb=(814, 3, 1536) | candidate_logits=14 | elapsed=146.9s


In [5]:
# Cell 5 - Batch Extraction Official BirdNET v2.4 (Logit 4-d Summary)
import hashlib
import json
import os
import subprocess
import sys
import time
from pathlib import Path
import numpy as np
from tqdm import tqdm

t0_bnet = time.time()

# 1. Pustaka Resmi birdnet & Load Model (v2.4 LiteRT)
try:
    import birdnet
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "birdnet"], check=True)
    import birdnet

try:
    bnet_model = birdnet.load("acoustic", "2.4", "tf", library="litert")
except Exception as e:
    print(f"litert fallback to tf: {e}")
    bnet_model = birdnet.load("acoustic", "2.4", "tf")

species_list = list(bnet_model.species_list)
N_SP = len(species_list)
SR_BNET = 48000

# 2. Resolusi Taksa Target (cocokkan lewat nama ilmiah)
sci_of = [s.replace("_", " ").split(" ")[0] + " " + s.replace("_", " ").split(" ")[1]
          if len(s.replace("_", " ").split(" ")) > 1 else s for s in species_list]
genus_of = [s.replace("_", " ").split(" ")[0] for s in species_list]

def find_species(sci_name):
    hits = [i for i, n in enumerate(sci_of) if n.lower() == sci_name.lower()]
    assert len(hits) == 1, f"'{sci_name}': diharapkan 1 taksa di BirdNET, diperoleh {len(hits)}"
    return hits[0]

IDX_BONDOL = find_species("Lonchura leucogastroides")
IDX_KUTILANG = find_species("Pycnonotus aurigaster")

PARADISAEIDAE_GENERA = {
    "Astrapia", "Cicinnurus", "Drepanornis", "Epimachus", "Loboparadisea", "Lophorina",
    "Lycocorax", "Macgregoria", "Manucodia", "Paradigalla", "Paradisaea", "Parotia",
    "Phonygammus", "Pteridophora", "Ptiloris", "Seleucidis", "Semioptera",
}
CENDRA_IDX = [i for i, g in enumerate(genus_of) if g in PARADISAEIDAE_GENERA]
assert len(CENDRA_IDX) > 0, "Tidak ada taksa Paradisaeidae ditemukan di BirdNET"

print(f"birdnet_taxa_resolved | total={N_SP} | Bondol={IDX_BONDOL} | Kutilang={IDX_KUTILANG} | Cendra={len(CENDRA_IDX)}")
for i in CENDRA_IDX:
    print(f"  Cendra -> [{i}] {species_list[i]}")
if len(CENDRA_IDX) != 17:
    print(f"  [warn] Jumlah taksa Cendrawasih = {len(CENDRA_IDX)} (bukan 17). Cek daftar di atas.")

# Spesies Perch (Cell 4) yang tidak ada di BirdNET, hanya informasi
try:
    missing_in_bnet = [s for s in CENDRA_SCI if s.lower() not in {n.lower() for n in sci_of}]
    if missing_in_bnet:
        print(f"  [info] Ada di Perch tapi tidak di BirdNET: {missing_in_bnet}")
except NameError:
    pass

NONTARGET = np.ones(N_SP, dtype=bool)
NONTARGET[[IDX_BONDOL, IDX_KUTILANG] + CENDRA_IDX] = False

# 3. Pemetaan Matriks Logit Penuh (dengan penjaga id tidak valid)
MISSING_MAX = 0

def to_full_matrix(res, n_inputs: int) -> np.ndarray:
    global MISSING_MAX
    full = np.full((n_inputs, N_SP), fill_value=-50.0, dtype=np.float32)
    ids = np.asarray(res.species_ids)[:, 0, :]
    vals = np.asarray(res.species_probs)[:, 0, :]  # logits saat apply_sigmoid=False
    for r in range(n_inputs):
        ok = (ids[r] >= 0) & (ids[r] < N_SP)
        full[r, ids[r][ok]] = vals[r][ok]
        MISSING_MAX = max(MISSING_MAX, N_SP - int(ok.sum()))
    return full

# 4. Ekstraksi Batch + Checkpoint beridentitas
CHUNK_FILES = 64
CKPT_DIR = WORK_DIR / "birdnet_ckpt"
CKPT_DIR.mkdir(exist_ok=True)

cfg_sig = hashlib.md5(json.dumps(
    {"n_sp": N_SP, "b": IDX_BONDOL, "k": IDX_KUTILANG, "c": CENDRA_IDX, "chunk": CHUNK_FILES}
).encode()).hexdigest()[:8]

# Fungsi peredam stderr native (didefinisikan di Cell 4); fallback no-op kalau belum ada
try:
    silence_native_stderr
except NameError:
    import contextlib
    silence_native_stderr = contextlib.nullcontext

def extract_birdnet_split_from_ram(arr_48k: np.ndarray, tag: str):
    n_files = len(arr_48k)
    out = np.full((n_files, 3, 4), np.nan, dtype=np.float32)
    data_sig = hashlib.md5(np.ascontiguousarray(arr_48k[:, 1, ::997]).tobytes()).hexdigest()[:8]

    for s in tqdm(range(0, n_files, CHUNK_FILES), desc=f"BirdNET {tag}"):
        chunk_len = min(CHUNK_FILES, n_files - s)
        ck = CKPT_DIR / f"{tag}_{cfg_sig}_{data_sig}_{s:06d}.npy"

        if ck.exists():
            try:
                blk = np.load(ck)
                if blk.shape == (chunk_len, 3, 4) and np.isfinite(blk).all():
                    out[s : s + chunk_len] = blk
                    continue
            except Exception:
                pass  # checkpoint rusak, hitung ulang

        inputs = [(arr_48k[i, w], SR_BNET) for i in range(s, s + chunk_len) for w in range(3)]

        with silence_native_stderr():
            res = bnet_model.predict_arrays(
                inputs, top_k=N_SP, apply_sigmoid=False, default_confidence_threshold=-1e9
            )
        full = to_full_matrix(res, len(inputs)).reshape(chunk_len, 3, N_SP)

        blk = np.stack(
            [
                full[:, :, IDX_BONDOL],
                full[:, :, IDX_KUTILANG],
                full[:, :, CENDRA_IDX].max(axis=-1),
                full[:, :, NONTARGET].max(axis=-1),
            ],
            axis=-1,
        ).astype(np.float32)

        tmp = ck.with_name(ck.stem + ".tmp.npy")
        np.save(tmp, blk)
        os.replace(tmp, ck)  # atomik
        out[s : s + chunk_len] = blk

    return out

print("extracting_birdnet_train...")
train_birdnet_sum = extract_birdnet_split_from_ram(train_48k, "train")
print("extracting_birdnet_test...")
test_birdnet_sum = extract_birdnet_split_from_ram(test_48k, "test")

# 5. Validasi dulu, baru simpan
n_tr, n_te = len(train_df), len(sub_df)
assert train_birdnet_sum.shape == (n_tr, 3, 4) and test_birdnet_sum.shape == (n_te, 3, 4)
assert np.isfinite(train_birdnet_sum).all() and np.isfinite(test_birdnet_sum).all(), "NaN/Inf di ringkasan BirdNET"
if MISSING_MAX > 0:
    print(f"  [warn] Ada input dengan {MISSING_MAX} spesies tanpa skor (terisi -50.0)")

np.save(WORK_DIR / "birdnet_train_summary.npy", train_birdnet_sum)
np.save(WORK_DIR / "birdnet_test_summary.npy", test_birdnet_sum)

# 6. Cek akal: rata-rata fitur per kelas [Bondol, Kutilang, Cendra_max, NT_max]
print("sanity_mean_per_class (kolom: Bondol | Kutilang | Cendra_max | NT_max)")
for lab in sorted(train_df["label"].unique()):
    m = train_birdnet_sum[(train_df["label"] == lab).values].mean(axis=(0, 1))
    print(f"  {lab:<12} {np.round(m, 2)}")

elapsed_c5 = time.time() - t0_bnet
print(
    f"birdnet_extracted | train_sum={train_birdnet_sum.shape} | "
    f"test_sum={test_birdnet_sum.shape} | nan=0 | elapsed={elapsed_c5:.1f}s"
)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.6/211.6 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.8/21.8 MB 71.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 74.2 MB/s eta 0:00:00


birdnet_taxa_resolved | total=6522 | Bondol=3260 | Kutilang=5133 | Cendra=17
  Cendra -> [1312] Cicinnurus magnificus_Magnificent Bird-of-Paradise
  Cendra -> [1313] Cicinnurus regius_King Bird-of-Paradise
  Cendra -> [1314] Cicinnurus respublica_Wilson's Bird-of-Paradise
  Cendra -> [1955] Drepanornis albertisi_Black-billed Sicklebill
  Cendra -> [2157] Epimachus fastosus_Black Sicklebill
  Cendra -> [2158] Epimachus meyeri_Brown Sicklebill
  Cendra -> [3277] Lophorina superba_Greater Lophorina
  Cendra -> [3314] Lycocorax pyrrhopterus_Paradise-crow
  Cendra -> [4218] Paradisaea minor_Lesser Bird-of-Paradise
  Cendra -> [4219] Paradisaea rubra_Red Bird-of-Paradise
  Cendra -> [4485] Phonygammus keraudrenii_Trumpet Manucode
  Cendra -> [5029] Pteridophora alberti_King-of-Saxony Bird-of-Paradise
  Cendra -> [5109] Ptiloris magnificus_Magnificent Riflebird
  Cendra -> [5110] Ptiloris paradiseus_Paradise Riflebird
  Cendra -> [5111] Ptiloris victoriae_Victoria's Riflebird
  Cendra -> [548

BirdNET train: 100%|██████████| 29/29 [04:58<00:00, 10.28s/it]


extracting_birdnet_test...


BirdNET test: 100%|██████████| 13/13 [02:12<00:00, 10.21s/it]

sanity_mean_per_class (kolom: Bondol | Kutilang | Cendra_max | NT_max)
  Bondol Jawa  [-14.07 -11.72 -10.24  -1.75]
  Cendrawasih  [-14.08 -11.6   -6.42  -2.14]
  Kutilang     [-15.2   -3.92 -10.19  -1.57]
birdnet_extracted | train_sum=(1826, 3, 4) | test_sum=(814, 3, 4) | nan=0 | elapsed=484.5s


In [6]:
# Cell 6 - Explicit Model Deletion, VRAM Purge, & Memory Sanity Check
import gc
import json
import os
from pathlib import Path
import numpy as np
import pandas as pd
import psutil
import torch
import tensorflow as tf

# 1. Hapus objek model berat & buffer audio mentah dari RAM
heavy_vars = [
    "perch_model", "infer_fn", "bnet_model", "dummy_out", "flat_wavs",
    "train_32k", "test_32k", "train_48k", "test_48k",
]
freed = []
for var_name in heavy_vars:
    if var_name in globals():
        del globals()[var_name]
        freed.append(var_name)

# 2. Bersihkan session backend & VRAM (memori GPU milik TF tidak dikembalikan ke OS)
tf.keras.backend.clear_session()
gc.collect()
purge_vram()

# 3. Verifikasi artefak fitur di disk (shape, finite, dan keselarasan baris)
n_tr, n_te = len(train_df), len(sub_df)
with open(WORK_DIR / "perch_taxa_indices.json") as f:
    summary_dim = int(json.load(f)["summary_dim"])

expected_artifacts = [
    ("perch_v2_train_emb.npy", (n_tr, 3, 1536)),
    ("perch_v2_test_emb.npy", (n_te, 3, 1536)),
    ("train_raw_cand_summary.npy", (n_tr, 3, summary_dim)),
    ("test_raw_cand_summary.npy", (n_te, 3, summary_dim)),
    ("birdnet_train_summary.npy", (n_tr, 3, 4)),
    ("birdnet_test_summary.npy", (n_te, 3, 4)),
]

for fname, exp_shape in expected_artifacts:
    fpath = WORK_DIR / fname
    assert fpath.exists(), f"Artefak wajib hilang: {fname}"
    arr_meta = np.load(fpath, mmap_mode="r")
    assert arr_meta.shape == exp_shape, f"Dimensi {fname} salah: {arr_meta.shape} vs {exp_shape}"
    assert np.isfinite(arr_meta).all(), f"NaN/Inf terdeteksi di {fname}"

folds_path = WORK_DIR / "metadata_with_folds.csv"
assert folds_path.exists(), "Artefak wajib hilang: metadata_with_folds.csv"
saved_meta = pd.read_csv(folds_path)
assert {"file_id", "fold"}.issubset(saved_meta.columns), "Kolom file_id/fold tidak ada di metadata_with_folds.csv"
assert saved_meta["file_id"].tolist() == train_df["file_id"].tolist(), \
    "Urutan file_id di metadata_with_folds.csv berbeda dari train_df (fitur akan tidak sejajar)"
assert (saved_meta["fold"].values == train_df["fold"].values).all(), "Fold di disk berbeda dari train_df"

# 4. Status memori RAM & VRAM
ram_used_gb = psutil.Process(os.getpid()).memory_info().rss / (1024**3)
ram_sys_gb = psutil.virtual_memory().used / (1024**3)

vram_report = []
if torch.cuda.is_available():
    for d in range(torch.cuda.device_count()):
        free_b, total_b = torch.cuda.mem_get_info(d)
        vram_report.append(f"cuda:{d}={free_b / 1024**3:.2f}/{total_b / 1024**3:.2f}GB")

print(
    f"memory_sanitized | freed={len(freed)} vars | artifacts_ok={len(expected_artifacts) + 1} | "
    f"proc_ram={ram_used_gb:.2f}GB | sys_ram={ram_sys_gb:.2f}GB | "
    f"vram_free/total: {' '.join(vram_report) if vram_report else 'n/a'} | "
    f"pytorch_cuda_ready={torch.cuda.is_available()}"
)

memory_sanitized | freed=8 vars | artifacts_ok=7 | proc_ram=3.30GB | sys_ram=2.90GB | vram_free/total: cuda:0=7.34/14.56GB cuda:1=14.46/14.56GB | pytorch_cuda_ready=True


In [7]:
# Cell 7 - Per-Fold Taxa Verification (7-d Summary -> 1543-d Hybrid Input) & Supervised Cosine Head
import json
import time
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix, f1_score, roc_auc_score
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

t0_c7 = time.time()

SEED = 42
N_EPOCHS_HEAD = 40
BATCH_SIZE_HEAD = 32
EMB_DIM = 256
NUM_CLASSES = 3
F1_EXPECTED = 0.9250   # hanya peringatan, bukan assert

CLASS_TO_IDX = {"Bondol Jawa": 0, "Cendrawasih": 1, "Kutilang": 2}
IDX_TO_CLASS = {v: k for k, v in CLASS_TO_IDX.items()}
assert set(train_df["label"].unique()) <= set(CLASS_TO_IDX), f"Label tak dikenal: {set(train_df['label'].unique())}"
train_df["target"] = train_df["label"].map(CLASS_TO_IDX).astype(int)

y_target = train_df["target"].values
folds = train_df["fold"].values
fold_ids = np.unique(folds)
N_FOLDS = len(fold_ids)
n_tr, n_te = len(train_df), len(sub_df)

# 1. Muat fitur dan konfigurasi taksa
with open(WORK_DIR / "perch_taxa_indices.json") as f:
    taxa_meta = json.load(f)

cendra_sp_names = list(taxa_meta["cendra"].keys())
n_cand = len(cendra_sp_names)

train_raw_sum = np.load(WORK_DIR / "train_raw_cand_summary.npy")
test_raw_sum = np.load(WORK_DIR / "test_raw_cand_summary.npy")
X_tr_emb = np.load(WORK_DIR / "perch_v2_train_emb.npy")
X_te_emb = np.load(WORK_DIR / "perch_v2_test_emb.npy")
assert train_raw_sum.shape[0] == n_tr and X_tr_emb.shape[0] == n_tr, "Fitur train tidak sejajar dengan train_df"
assert test_raw_sum.shape[0] == n_te and X_te_emb.shape[0] == n_te, "Fitur test tidak sejajar dengan sub_df"
assert train_raw_sum.shape[-1] == taxa_meta["summary_dim"]

COL_B, COL_K = 0, 1
cendra_cols = list(range(2, 2 + n_cand))
COL_MAX_NT = 2 + n_cand
COL_TOP5_NT = 2 + n_cand + 1
COL_ENERGY = 2 + n_cand + 2

N_SUMMARY = 7
IN_FEATURES = X_tr_emb.shape[-1] + N_SUMMARY   # 1536 + 7 = 1543
assert IN_FEATURES == 1543, f"IN_FEATURES={IN_FEATURES}"

# 2. Seleksi taksa Cendrawasih: HANYA dari baris train tiap fold
y_cendra_all = (y_target == 1).astype(int)
cand_max = np.max(train_raw_sum[:, :, 2:2 + n_cand], axis=1)   # (N, n_cand): max antar window

def safe_auc(y_true, scores):
    return float(roc_auc_score(y_true, scores)) if len(np.unique(y_true)) == 2 else np.nan

def pick_cendra(tr_mask):
    aucs = np.array([safe_auc(y_cendra_all[tr_mask], cand_max[tr_mask, j]) for j in range(n_cand)])
    return int(np.nanargmax(aucs)), aucs

# 3. Rakit ringkasan 7-d & input hybrid 1543-d
def build_summary_7d(raw_sum, c_col):
    z_b = raw_sum[:, :, COL_B:COL_B + 1]
    z_k = raw_sum[:, :, COL_K:COL_K + 1]
    z_c = raw_sum[:, :, c_col:c_col + 1]
    max_nt = raw_sum[:, :, COL_MAX_NT:COL_MAX_NT + 1]
    top5_nt = raw_sum[:, :, COL_TOP5_NT:COL_TOP5_NT + 1]
    energy = raw_sum[:, :, COL_ENERGY:COL_ENERGY + 1]
    margin_ood = max_nt - np.maximum(np.maximum(z_b, z_k), z_c)
    return np.concatenate([z_b, z_c, z_k, max_nt, top5_nt, margin_ood, energy], axis=2).astype(np.float32)

def fit_zscore(s7_train):
    mu = s7_train.mean(axis=(0, 1), keepdims=True)
    sd = s7_train.std(axis=(0, 1), keepdims=True) + 1e-6
    return mu, sd

def make_hybrid(raw, emb, c_col, mu, sd):
    s7 = (build_summary_7d(raw, c_col) - mu) / sd
    return np.concatenate([emb, s7], axis=-1).astype(np.float32)

# 4. Arsitektur
class CosineLinear(nn.Module):
    def __init__(self, in_features, num_classes=3, init_tau=16.0):
        super().__init__()
        self.weight = nn.Parameter(torch.randn(num_classes, in_features))
        nn.init.xavier_uniform_(self.weight)
        self.tau = nn.Parameter(torch.tensor(init_tau, dtype=torch.float32))

    def forward(self, x):
        x_norm = F.normalize(x, p=2, dim=-1)
        w_norm = F.normalize(self.weight, p=2, dim=-1)
        return torch.clamp(self.tau, min=1.0, max=32.0) * F.linear(x_norm, w_norm)

class PerchBottleneckCosineHead(nn.Module):
    def __init__(self, in_features=IN_FEATURES, emb_dim=EMB_DIM, num_classes=NUM_CLASSES, dropout=0.2, init_tau=16.0):
        super().__init__()
        self.bottleneck = nn.Sequential(
            nn.Linear(in_features, emb_dim, bias=False),
            nn.LayerNorm(emb_dim),
            nn.SiLU(inplace=True),
            nn.Dropout(p=dropout),
        )
        self.classifier = CosineLinear(emb_dim, num_classes=num_classes, init_tau=init_tau)

    def forward(self, x_3w):
        B, W, D = x_3w.shape
        h = self.bottleneck(x_3w.reshape(B * W, D))
        h_norm = F.normalize(h, p=2, dim=-1)
        logits_3w = self.classifier(h_norm).view(B, W, -1)
        h_3w = h_norm.view(B, W, -1)
        conf_weights = F.softmax(logits_3w.max(dim=-1, keepdim=True)[0] * 2.0, dim=1)
        bag_logits = (logits_3w * conf_weights).sum(dim=1)
        bag_emb = F.normalize((h_3w * conf_weights).sum(dim=1), p=2, dim=-1)
        return bag_logits, bag_emb

# 5. Class-Balanced Focal Loss
def compute_effective_num_weights(class_counts, beta=0.99):
    counts = np.asarray(class_counts, dtype=np.float64)
    weights = (1.0 - beta) / (1.0 - np.power(beta, counts))
    weights = weights / np.sum(weights) * len(counts)
    return torch.tensor(weights, dtype=torch.float32)

class ClassBalancedFocalLoss(nn.Module):
    def __init__(self, class_counts, beta=0.99, gamma=2.0, smoothing=0.05):
        super().__init__()
        self.gamma = gamma
        self.smoothing = smoothing
        self.num_classes = len(class_counts)
        self.register_buffer("alpha", compute_effective_num_weights(class_counts, beta=beta))

    def forward(self, logits, targets):
        log_probs = F.log_softmax(logits, dim=-1)
        probs = torch.exp(log_probs)
        if targets.dim() == 1:
            with torch.no_grad():
                true_dist = torch.zeros_like(logits)
                true_dist.fill_(self.smoothing / (self.num_classes - 1))
                true_dist.scatter_(1, targets.unsqueeze(1), 1.0 - self.smoothing)
        else:
            true_dist = targets * (1.0 - self.smoothing) + (self.smoothing / self.num_classes)
        focal_weight = torch.pow(1.0 - probs, self.gamma)
        loss = -torch.sum(self.alpha.unsqueeze(0) * focal_weight * true_dist * log_probs, dim=-1)
        return torch.mean(loss)

# 6. Pelatihan 4-fold (seleksi taksa & z-score per fold, tanpa peeking)
oof_probs = np.zeros((n_tr, NUM_CLASSES), dtype=np.float32)
oof_emb256 = np.zeros((n_tr, EMB_DIM), dtype=np.float32)
test_probs = np.zeros((n_te, NUM_CLASSES), dtype=np.float32)
test_emb256_per_fold = np.zeros((N_FOLDS, n_te, EMB_DIM), dtype=np.float32)

fold_winner_info = {}
fold_final_f1s = []

print(f"training_supervised_hybrid_heads ({N_EPOCHS_HEAD} epochs per fold, seleksi taksa per fold)...")
for k, fold in enumerate(fold_ids):
    t_f = time.time()
    torch.manual_seed(SEED + int(fold))
    np.random.seed(SEED + int(fold))

    tr_mask = folds != fold
    tr_idx = np.where(tr_mask)[0]
    val_idx = np.where(~tr_mask)[0]

    # seleksi taksa + z-score hanya dari baris train fold ini
    j, aucs_tr = pick_cendra(tr_mask)
    c_col = cendra_cols[j]
    mu, sd = fit_zscore(build_summary_7d(train_raw_sum[tr_idx], c_col))
    held_auc = safe_auc(y_cendra_all[val_idx], cand_max[val_idx, j])
    fold_winner_info[int(fold)] = {"species": cendra_sp_names[j], "col": int(c_col),
                                   "in_fold_auroc": float(aucs_tr[j]), "heldout_auroc": held_auc}

    x_tr_np = make_hybrid(train_raw_sum[tr_idx], X_tr_emb[tr_idx], c_col, mu, sd)
    x_va_np = make_hybrid(train_raw_sum[val_idx], X_tr_emb[val_idx], c_col, mu, sd)
    x_te_np = make_hybrid(test_raw_sum, X_te_emb, c_col, mu, sd)
    assert np.isfinite(x_tr_np).all() and np.isfinite(x_va_np).all() and np.isfinite(x_te_np).all()

    x_tr = torch.from_numpy(x_tr_np)
    y_tr = torch.from_numpy(y_target[tr_idx]).long()
    x_va = torch.from_numpy(x_va_np).to(DEVICE)
    x_te = torch.from_numpy(x_te_np).to(DEVICE)
    y_va = y_target[val_idx]

    gen = torch.Generator().manual_seed(SEED + int(fold))
    dl_tr = DataLoader(TensorDataset(x_tr, y_tr), batch_size=BATCH_SIZE_HEAD, shuffle=True, generator=gen)

    model = PerchBottleneckCosineHead().to(DEVICE)
    decay, no_decay = [], []
    for _, p in model.named_parameters():
        (no_decay if p.ndim <= 1 else decay).append(p)
    optimizer = torch.optim.AdamW(
        [{"params": decay, "weight_decay": 1e-4}, {"params": no_decay, "weight_decay": 0.0}], lr=1e-3,
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=N_EPOCHS_HEAD, eta_min=1e-5)

    tr_counts = [int(np.sum(y_target[tr_idx] == c)) for c in range(NUM_CLASSES)]
    criterion = ClassBalancedFocalLoss(tr_counts, beta=0.99, gamma=2.0).to(DEVICE)

    for ep in range(1, N_EPOCHS_HEAD + 1):
        model.train()
        for bx, by in dl_tr:
            bx, by = bx.to(DEVICE), by.to(DEVICE)
            optimizer.zero_grad()
            logits, _ = model(bx)
            criterion(logits, by).backward()
            optimizer.step()
        scheduler.step()

    model.eval()
    with torch.no_grad():
        v_logits, v_emb = model(x_va)
        v_probs = F.softmax(v_logits, dim=-1).cpu().numpy()
        v_emb = v_emb.cpu().numpy()
        t_logits, t_emb = model(x_te)
        t_probs = F.softmax(t_logits, dim=-1).cpu().numpy()
        t_emb = t_emb.cpu().numpy()

    v_pred = np.argmax(v_probs, axis=1)
    final_f1 = float(f1_score(y_va, v_pred, average="macro"))
    cendra_f1 = float(f1_score(y_va == 1, v_pred == 1, zero_division=0))
    fold_final_f1s.append(final_f1)

    oof_probs[val_idx] = v_probs
    oof_emb256[val_idx] = v_emb
    test_probs += t_probs / N_FOLDS
    test_emb256_per_fold[k] = t_emb
    torch.save(model.state_dict(), WORK_DIR / f"head_supervised_fold{int(fold)}.pt")

    print(f"  fold={k + 1}/{N_FOLDS} | cendra_sp='{cendra_sp_names[j]}' (in_auc={aucs_tr[j]:.3f}, held_auc={held_auc:.3f}) | "
          f"val={len(val_idx):<4} | n_cendra={int(np.sum(y_va == 1)):<2} | macro_f1={final_f1:.4f} | "
          f"cendra_f1={cendra_f1:.4f} | {time.time() - t_f:.1f}s")

    del model, optimizer, x_va, x_te, x_tr

# 7. Evaluasi OOF global
oof_preds = np.argmax(oof_probs, axis=1)
global_macro_f1 = f1_score(y_target, oof_preds, average="macro")
per_class_f1 = f1_score(y_target, oof_preds, average=None)
cm = confusion_matrix(y_target, oof_preds, labels=[0, 1, 2])

print(f"\nsupervised_head_trained | input_dim={IN_FEATURES} | Global OOF Macro F1 = {global_macro_f1:.4f} | "
      f"mean_fold_f1={np.mean(fold_final_f1s):.4f} +/- {np.std(fold_final_f1s):.4f}")
print(f"Confusion Matrix (row=true, col=pred):\n{cm}")
for idx, name in IDX_TO_CLASS.items():
    print(f"  {name:<12} OOF F1: {per_class_f1[idx]:.4f}")
print(f"  spesies pemenang per fold: {[v['species'] for v in fold_winner_info.values()]}")

# 8. Simpan artefak DULU, baru peringatan
np.save(WORK_DIR / "oof_p_supervised.npy", oof_probs)
np.save(WORK_DIR / "test_p_supervised.npy", test_probs)
np.save(WORK_DIR / "oof_emb_supervised_256d.npy", oof_emb256)
np.save(WORK_DIR / "test_emb_supervised_256d_per_fold.npy", test_emb256_per_fold)
np.save(WORK_DIR / "test_emb_supervised_256d.npy", test_emb256_per_fold.mean(axis=0))  # kompatibilitas; ruang antar fold tidak selaras
with open(WORK_DIR / "supervised_fold_winners.json", "w") as f:
    json.dump(fold_winner_info, f, indent=2)

if global_macro_f1 < F1_EXPECTED:
    print(f"[warn] OOF macro F1 {global_macro_f1:.4f} di bawah standar baseline {F1_EXPECTED:.4f}")

# Untuk cell berikutnya yang butuh satu set fitur global: seleksi memakai seluruh train (bukan untuk skor validasi)
j_all, _ = pick_cendra(np.ones(n_tr, dtype=bool))
mu_all, sd_all = fit_zscore(build_summary_7d(train_raw_sum, cendra_cols[j_all]))
train_hybrid_1543 = make_hybrid(train_raw_sum, X_tr_emb, cendra_cols[j_all], mu_all, sd_all)
test_hybrid_1543 = make_hybrid(test_raw_sum, X_te_emb, cendra_cols[j_all], mu_all, sd_all)
assert train_hybrid_1543.shape == (n_tr, 3, IN_FEATURES) and np.isfinite(train_hybrid_1543).all()
np.save(WORK_DIR / "train_hybrid_1543.npy", train_hybrid_1543)
np.save(WORK_DIR / "test_hybrid_1543.npy", test_hybrid_1543)
print(f"global_locked_for_final_fit | species='{cendra_sp_names[j_all]}' (dipakai hanya untuk fit akhir, bukan validasi)")

print(f"cell7_completed | global_macro_f1={global_macro_f1:.4f} | elapsed={time.time() - t0_c7:.1f}s")

training_supervised_hybrid_heads (40 epochs per fold, seleksi taksa per fold)...
  fold=1/4 | cendra_sp='Cicinnurus regius' (in_auc=0.891, held_auc=0.890) | val=503  | n_cendra=35 | macro_f1=0.9836 | cendra_f1=0.9552 | 10.6s
  fold=2/4 | cendra_sp='Cicinnurus regius' (in_auc=0.850, held_auc=0.972) | val=462  | n_cendra=29 | macro_f1=1.0000 | cendra_f1=1.0000 | 8.8s
  fold=3/4 | cendra_sp='Paradisaea rubra' (in_auc=0.931, held_auc=0.739) | val=422  | n_cendra=16 | macro_f1=0.8539 | cendra_f1=0.5833 | 8.9s
  fold=4/4 | cendra_sp='Paradisaea rubra' (in_auc=0.908, held_auc=0.576) | val=439  | n_cendra=3  | macro_f1=0.6815 | cendra_f1=0.1111 | 8.9s

supervised_head_trained | input_dim=1543 | Global OOF Macro F1 = 0.9119 | mean_fold_f1=0.8798 +/- 0.1277
Confusion Matrix (row=true, col=pred):
[[1092   32    0]
 [  11   70    2]
 [   0    0  619]]
  Bondol Jawa  OOF F1: 0.9807
  Cendrawasih  OOF F1: 0.7568
  Kutilang     OOF F1: 0.9984
  spesies pemenang per fold: ['Cicinnurus regius', 'Cicinn

In [8]:
# Cell 8 - Nested In-Fold Threshold Selection (Cendrawasih & Kutilang) + Held-Out Check
import json
import time
from pathlib import Path
import numpy as np
from sklearn.metrics import roc_curve

t0_c8 = time.time()

# 1. Muat ringkasan BirdNET [Bondol, Kutilang, Cendra_max, NT_max] dan cek keselarasan
train_bnet = np.load(WORK_DIR / "birdnet_train_summary.npy")
assert train_bnet.shape == (len(train_df), 3, 4), f"Shape tidak sejajar: {train_bnet.shape}"
assert np.isfinite(train_bnet).all()

cen_tr = train_bnet[:, :, 2].max(axis=1)   # max Cendra antar window
kut_tr = train_bnet[:, :, 1].max(axis=1)   # logit Kutilang, max antar window

y_target = train_df["target"].values
folds = train_df["fold"].values
fold_ids = np.unique(folds)
N_FOLDS = len(fold_ids)

# 2. Helper
def youden_threshold(y_pos, scores):
    fpr, tpr, thr = roc_curve(y_pos.astype(int), scores)
    ok = np.isfinite(thr)
    j = int(np.argmax((tpr - fpr)[ok]))
    return float(thr[ok][j])

def rates(y_pos, scores, thr):
    """TPR dan FPR untuk aturan 'skor >= thr'."""
    pred = scores >= thr
    tpr = float(pred[y_pos].mean()) if y_pos.any() else np.nan
    fpr = float(pred[~y_pos].mean()) if (~y_pos).any() else np.nan
    return tpr, fpr

def compute_thresholds(mask):
    pos_c = y_target[mask] == 1
    pos_k = y_target[mask] == 2
    return {
        "cendra_p80": float(np.percentile(cen_tr[mask][~pos_c], 80.0)),
        "cendra_youden": youden_threshold(pos_c, cen_tr[mask]),
        "kutilang_p95": float(np.percentile(kut_tr[mask][~pos_k], 95.0)),
        "kutilang_youden": youden_threshold(pos_k, kut_tr[mask]),
    }

# 3. Ambang per fold (dari baris train fold) + evaluasi di fold held-out
nested_thresholds = {}
print("computing_nested_in_fold_thresholds...")
print(f"{'fold':<5}{'ambang':<18}{'nilai':>8}   held-out TPR / FPR")
for f_id in fold_ids:
    tr_mask = folds != f_id
    va_mask = ~tr_mask
    thr = compute_thresholds(tr_mask)

    va_c = y_target[va_mask] == 1
    va_k = y_target[va_mask] == 2
    held = {}
    for name, thr_v in thr.items():
        if name.startswith("cendra"):
            held[name] = rates(va_c, cen_tr[va_mask], thr_v)
        else:
            held[name] = rates(va_k, kut_tr[va_mask], thr_v)
        print(f"{int(f_id):<5}{name:<18}{thr_v:>8.2f}   {held[name][0]:.3f} / {held[name][1]:.3f}")

    nested_thresholds[int(f_id)] = {
        "thr_cendra_p80": thr["cendra_p80"],
        "thr_cendra_youden": thr["cendra_youden"],
        "thr_kutilang_p95": thr["kutilang_p95"],
        "thr_kutilang_youden": thr["kutilang_youden"],
        "chosen_cendra": thr["cendra_p80"],
        "chosen_kutilang": thr["kutilang_p95"],
        "heldout_tpr_fpr": {k: [None if np.isnan(x) else x for x in v] for k, v in held.items()},
        "n_heldout_cendra": int(va_c.sum()),
        "n_heldout_kutilang": int(va_k.sum()),
    }

mean_thr_c = float(np.mean([v["chosen_cendra"] for v in nested_thresholds.values()]))
mean_thr_k = float(np.mean([v["chosen_kutilang"] for v in nested_thresholds.values()]))

# 4. Ambang final dari SELURUH train (untuk inferensi test, bukan untuk skor validasi)
all_mask = np.ones(len(train_df), dtype=bool)
final_thr = compute_thresholds(all_mask)

# 5. Simpan DULU
meta_thr = {
    "fold_thresholds": nested_thresholds,
    "global_mean_thr_cendra": mean_thr_c,
    "global_mean_thr_kutilang": mean_thr_k,
    "final_thr_cendra": final_thr["cendra_p80"],
    "final_thr_kutilang": final_thr["kutilang_p95"],
    "final_all_methods": final_thr,
    "cendra_confidence_floor": 0.60,
    "note": "nilai ambang dihitung in-fold; pemilihan metode (p80/p95) berasal dari eksplorasi sebelumnya",
}
with open(WORK_DIR / "nested_forensic_thresholds.json", "w") as fh:
    json.dump(meta_thr, fh, indent=2)

# 6. Ringkasan + peringatan (bukan assert)
print("\nnested_thresholds_locked:")
print(f"  Cendrawasih (p80) mean antar fold : {mean_thr_c:.2f} | final (seluruh train): {final_thr['cendra_p80']:.2f}")
print(f"  Kutilang    (p95) mean antar fold : {mean_thr_k:.2f} | final (seluruh train): {final_thr['kutilang_p95']:.2f}")
sd_c = float(np.std([v["chosen_cendra"] for v in nested_thresholds.values()]))
sd_k = float(np.std([v["chosen_kutilang"] for v in nested_thresholds.values()]))
print(f"  Std antar fold: cendra={sd_c:.2f} | kutilang={sd_k:.2f}")

for name, val, lo, hi in [("Cendrawasih", mean_thr_c, -8.50, -7.00), ("Kutilang", mean_thr_k, -8.80, -7.50)]:
    if not (lo <= val <= hi):
        print(f"  [warn] Ambang {name} {val:.2f} di luar rentang acuan eksperimen lama [{lo}, {hi}]")

print(f"thresholds_saved -> {WORK_DIR / 'nested_forensic_thresholds.json'}")
print(f"cell8_elapsed={time.time() - t0_c8:.2f}s")

computing_nested_in_fold_thresholds...
fold ambang               nilai   held-out TPR / FPR
0    cendra_p80           -7.66   0.886 / 0.171
0    cendra_youden        -7.34   0.886 / 0.145
0    kutilang_p95         -8.15   0.970 / 0.065
0    kutilang_youden      -7.37   0.955 / 0.016
1    cendra_p80           -7.70   1.000 / 0.201
1    cendra_youden        -7.34   0.966 / 0.171
1    kutilang_p95         -8.05   0.944 / 0.034
1    kutilang_youden      -7.34   0.916 / 0.006
2    cendra_p80           -7.68   0.812 / 0.185
2    cendra_youden        -7.24   0.688 / 0.131
2    kutilang_p95         -8.15   1.000 / 0.070
2    kutilang_youden      -7.37   1.000 / 0.016
3    cendra_p80           -7.77   1.000 / 0.239
3    cendra_youden        -7.25   1.000 / 0.154
3    kutilang_p95         -8.07   0.994 / 0.034
3    kutilang_youden      -7.37   0.994 / 0.011

nested_thresholds_locked:
  Cendrawasih (p80) mean antar fold : -7.70 | final (seluruh train): -7.70
  Kutilang    (p95) mean antar fold : 

In [9]:
# Cell 9 - Supervised Baseline OOF Reassignment Evaluation & Anchor Test Probability Matrix
import json
import time
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix, f1_score, precision_score, recall_score

t0_c9 = time.time()

CLASS_TO_IDX = {"Bondol Jawa": 0, "Cendrawasih": 1, "Kutilang": 2}
IDX_TO_CLASS = {v: k for k, v in CLASS_TO_IDX.items()}

# 1. Muat prediksi Cell 7, ambang Cell 8, dan logit BirdNET
oof_p = np.load(WORK_DIR / "oof_p_supervised.npy")
test_p = np.load(WORK_DIR / "test_p_supervised.npy")

with open(WORK_DIR / "nested_forensic_thresholds.json") as fh:
    thr_meta = json.load(fh)
fold_thrs = thr_meta["fold_thresholds"]
final_thr_cendra = thr_meta["final_thr_cendra"]

train_bnet = np.load(WORK_DIR / "birdnet_train_summary.npy")
test_bnet = np.load(WORK_DIR / "birdnet_test_summary.npy")

n_tr, n_te = len(train_df), len(sub_df)
assert oof_p.shape == (n_tr, 3) and test_p.shape == (n_te, 3), "Shape prediksi tidak sejajar"
assert train_bnet.shape[0] == n_tr and test_bnet.shape[0] == n_te, "Shape BirdNET tidak sejajar"

cen_oof = train_bnet[:, :, 2].max(axis=1)
cen_test = test_bnet[:, :, 2].max(axis=1)

y_true = train_df["target"].values
folds = train_df["fold"].values

fold_keys = {str(int(f)) for f in np.unique(folds)}
assert fold_keys <= set(fold_thrs), f"Kunci fold tidak ada di JSON: {fold_keys - set(fold_thrs)}"

# 2. Fungsi reassignment (dipakai untuk OOF dan test)
def reassign(p, score, thr):
    """Jika prediksi = Cendrawasih tapi skor BirdNET < ambang, pindahkan massa Cendra ke kelas kedua."""
    pred_raw = np.argmax(p, axis=1)
    second = np.argsort(p, axis=1)[:, -2]
    mask = (pred_raw == 1) & (score < thr)
    corr = pred_raw.copy()
    corr[mask] = second[mask]
    p2 = p.copy()
    idx = np.where(mask)[0]
    p2[idx, second[idx]] += p2[idx, 1]
    p2[idx, 1] = 0.0
    p2 /= p2.sum(axis=1, keepdims=True)   # jumlah sudah 1, hanya pengaman numerik
    return pred_raw, corr, p2, mask

# 3. Evaluasi OOF (ambang per fold dari baris train fold tersebut)
thr_oof = np.array([fold_thrs[str(int(f))]["chosen_cendra"] for f in folds])
pred_raw_oof, corr_pred_oof, oof_p_reassigned, mask_oof = reassign(oof_p, cen_oof, thr_oof)
oof_reassigned_count = int(mask_oof.sum())

f1_raw = float(f1_score(y_true, pred_raw_oof, average="macro"))
f1_corr = float(f1_score(y_true, corr_pred_oof, average="macro"))
f1_per_class_corr = f1_score(y_true, corr_pred_oof, average=None)
cm_raw = confusion_matrix(y_true, pred_raw_oof, labels=[0, 1, 2])
cm_corr = confusion_matrix(y_true, corr_pred_oof, labels=[0, 1, 2])

tp_lost = int((mask_oof & (y_true == 1)).sum())            # Cendrawasih asli yang ikut dibuang
fp_removed = int((mask_oof & (y_true != 1)).sum())         # false positive yang dibuang
fp_fixed = int((mask_oof & (y_true != 1) & (corr_pred_oof == y_true)).sum())  # ...dan jatuh ke kelas yang benar

rec_raw = float(recall_score(y_true == 1, pred_raw_oof == 1, zero_division=0))
rec_corr = float(recall_score(y_true == 1, corr_pred_oof == 1, zero_division=0))
prec_raw = float(precision_score(y_true == 1, pred_raw_oof == 1, zero_division=0))
prec_corr = float(precision_score(y_true == 1, corr_pred_oof == 1, zero_division=0))

print("oof_forensic_reassignment_audit:")
print(f"  Dipindahkan dari Cendrawasih       : {oof_reassigned_count} (FP dibuang={fp_removed}, di antaranya benar={fp_fixed} | TP hilang={tp_lost})")
print(f"  Cendrawasih recall / precision     : {rec_raw:.3f} / {prec_raw:.3f}  ->  {rec_corr:.3f} / {prec_corr:.3f}")
print(f"  Macro F1 sebelum                   : {f1_raw:.4f}")
print(f"  Macro F1 sesudah                   : {f1_corr:.4f} (delta={f1_corr - f1_raw:+.4f})")
print(f"  Per-class F1 sesudah               : Bondol={f1_per_class_corr[0]:.4f} | Cendra={f1_per_class_corr[1]:.4f} | Kutilang={f1_per_class_corr[2]:.4f}")
print(f"Confusion Matrix sebelum (row=true, col=pred):\n{cm_raw}")
print(f"Confusion Matrix sesudah:\n{cm_corr}")

# 4. Test set (ambang dari seluruh train)
pred_raw_test, corr_pred_test, test_p_reassigned, mask_test = reassign(test_p, cen_test, final_thr_cendra)
test_reassigned_count = int(mask_test.sum())

print("\ntest_anchor_reassignment_applied:")
print(f"  Test Cendrawasih calls raw         : {int((pred_raw_test == 1).sum())}")
print(f"  Test Cendrawasih calls reassigned  : {test_reassigned_count}")
print(f"  Test Cendrawasih calls remaining   : {int((corr_pred_test == 1).sum())}")
print(f"  Test class distribution raw        : {pd.Series([IDX_TO_CLASS[c] for c in pred_raw_test]).value_counts().to_dict()}")
print(f"  Test class distribution reassigned : {pd.Series([IDX_TO_CLASS[c] for c in corr_pred_test]).value_counts().to_dict()}")

# 5. Simpan artefak
np.save(WORK_DIR / "oof_p_supervised_reassigned.npy", oof_p_reassigned)
np.save(WORK_DIR / "test_p_supervised_anchor.npy", test_p_reassigned)

meta_c9 = {
    "oof_macro_f1_raw": f1_raw,
    "oof_macro_f1_reassigned": f1_corr,
    "oof_reassigned_samples": oof_reassigned_count,
    "oof_fp_removed": fp_removed,
    "oof_fp_fixed": fp_fixed,
    "oof_tp_lost": tp_lost,
    "oof_cendra_recall_raw": rec_raw,
    "oof_cendra_recall_reassigned": rec_corr,
    "oof_cendra_precision_raw": prec_raw,
    "oof_cendra_precision_reassigned": prec_corr,
    "test_cendra_raw": int((pred_raw_test == 1).sum()),
    "test_cendra_reassigned": test_reassigned_count,
    "test_cendra_clean": int((corr_pred_test == 1).sum()),
    "cm_raw": cm_raw.tolist(),
    "cm_reassigned": cm_corr.tolist(),
}
with open(WORK_DIR / "supervised_reassignment_meta.json", "w") as fh:
    json.dump(meta_c9, fh, indent=2)

print(f"\ncell9_completed | reassigned_oof_macro_f1={f1_corr:.4f} | elapsed={time.time() - t0_c9:.2f}s")

oof_forensic_reassignment_audit:
  Dipindahkan dari Cendrawasih       : 21 (FP dibuang=19, di antaranya benar=18 | TP hilang=2)
  Cendrawasih recall / precision     : 0.843 / 0.686  ->  0.819 / 0.840
  Macro F1 sebelum                   : 0.9119
  Macro F1 sesudah                   : 0.9383 (delta=+0.0263)
  Per-class F1 sesudah               : Bondol=0.9880 | Cendra=0.8293 | Kutilang=0.9976
Confusion Matrix sebelum (row=true, col=pred):
[[1092   32    0]
 [  11   70    2]
 [   0    0  619]]
Confusion Matrix sesudah:
[[1110   13    1]
 [  13   68    2]
 [   0    0  619]]

test_anchor_reassignment_applied:
  Test Cendrawasih calls raw         : 210
  Test Cendrawasih calls reassigned  : 84
  Test Cendrawasih calls remaining   : 126
  Test class distribution raw        : {'Bondol Jawa': 379, 'Kutilang': 225, 'Cendrawasih': 210}
  Test class distribution reassigned : {'Bondol Jawa': 429, 'Kutilang': 259, 'Cendrawasih': 126}

cell9_completed | reassigned_oof_macro_f1=0.9383 | elapsed=0.04s

In [10]:
# Cell 10 - kNN-5 Cosine Manifold Distance & Consensus Inlierness Curation
import json
import time
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.neighbors import NearestNeighbors

t0_c10 = time.time()
K_NEIGHBORS = 5
NUM_CLASSES = 3
P_PCT, D_PCT = 75.0, 40.0
KUT_CONF_FLOOR = 0.60

CLASS_TO_IDX = {"Bondol Jawa": 0, "Cendrawasih": 1, "Kutilang": 2}
IDX_TO_CLASS = {v: k for k, v in CLASS_TO_IDX.items()}

# 1. Muat fitur & cek keselarasan
X_tr_emb = np.load(WORK_DIR / "perch_v2_train_emb.npy")
X_te_emb = np.load(WORK_DIR / "perch_v2_test_emb.npy")
D = X_tr_emb.shape[-1]
n_tr, n_te = len(train_df), len(sub_df)
assert X_tr_emb.shape[0] == n_tr and X_te_emb.shape[0] == n_te, "Embedding tidak sejajar dengan metadata"

y_target = train_df["target"].values
folds = train_df["fold"].values
fold_ids = np.unique(folds)
N_FOLDS = len(fold_ids)

# 2. kNN-5 cosine in-fold (mean-centering dari train fold)
def center_norm(x, mu):
    x = x - mu
    return x / (np.linalg.norm(x, axis=-1, keepdims=True) + 1e-12)

def min_window_knn_dist(knn, emb_3w, mu):
    n, w, d = emb_3w.shape
    q = center_norm(emb_3w.reshape(n * w, d), mu)
    dists, _ = knn.kneighbors(q)
    return dists.mean(axis=1).reshape(n, w).min(axis=1)

oof_d_perch = np.zeros(n_tr, dtype=np.float32)
test_d_per_fold = np.zeros((N_FOLDS, n_te), dtype=np.float32)

print(f"computing_knn_distances (k={K_NEIGHBORS}, in-fold)...")
for k, fold in enumerate(fold_ids):
    tr_idx = np.where(folds != fold)[0]
    val_idx = np.where(folds == fold)[0]
    tr_w = X_tr_emb[tr_idx].reshape(-1, D)
    mu_fold = tr_w.mean(axis=0, keepdims=True)
    knn = NearestNeighbors(n_neighbors=K_NEIGHBORS, metric="cosine", n_jobs=-1).fit(center_norm(tr_w, mu_fold))
    oof_d_perch[val_idx] = min_window_knn_dist(knn, X_tr_emb[val_idx], mu_fold)
    test_d_per_fold[k] = min_window_knn_dist(knn, X_te_emb, mu_fold)

test_d_perch = test_d_per_fold.mean(axis=0).astype(np.float32)
assert np.isfinite(oof_d_perch).all() and np.isfinite(test_d_perch).all()
np.save(WORK_DIR / "oof_d_perch.npy", oof_d_perch)
np.save(WORK_DIR / "test_d_perch.npy", test_d_perch)

print(f"  oof_manifold_distance  : mean={oof_d_perch.mean():.4f} | p50={np.percentile(oof_d_perch, 50):.4f} | p90={np.percentile(oof_d_perch, 90):.4f}")
print(f"  test_manifold_distance : mean={test_d_perch.mean():.4f} | p50={np.percentile(test_d_perch, 50):.4f} | p90={np.percentile(test_d_perch, 90):.4f}")

# 3. Muat prediksi (sudah melewati reassignment Cell 9), ambang, dan BirdNET
test_p_anchor = np.load(WORK_DIR / "test_p_supervised_anchor.npy")
oof_p_reassigned = np.load(WORK_DIR / "oof_p_supervised_reassigned.npy")
assert test_p_anchor.shape == (n_te, NUM_CLASSES) and oof_p_reassigned.shape == (n_tr, NUM_CLASSES)

with open(WORK_DIR / "nested_forensic_thresholds.json") as fh:
    thr_meta = json.load(fh)
final_thr_cendra = thr_meta["final_thr_cendra"]
final_thr_kutilang = thr_meta["final_thr_kutilang"]
fold_thrs = thr_meta["fold_thresholds"]

test_bnet = np.load(WORK_DIR / "birdnet_test_summary.npy")
train_bnet = np.load(WORK_DIR / "birdnet_train_summary.npy")
cen_test, kut_test = test_bnet[:, :, 2].max(axis=1), test_bnet[:, :, 1].max(axis=1)
cen_oof, kut_oof = train_bnet[:, :, 2].max(axis=1), train_bnet[:, :, 1].max(axis=1)

# Skor NLL dari probabilitas kelas prediksi (BUKAN energi bebas Helmholtz; tidak dipakai untuk filter)
nll_maxp_test = -np.log(np.clip(test_p_anchor.max(axis=1), 1e-12, 1.0)).astype(np.float32)
np.save(WORK_DIR / "test_nll_maxp.npy", nll_maxp_test)

# 4. Aturan kurasi (vektorisasi, dipakai untuk test dan untuk diagnostik OOF)
def curate(p, d, cen, kut, thr_c, thr_k):
    pred = np.argmax(p, axis=1)
    maxp = p.max(axis=1)
    p_cut = float(np.percentile(maxp, P_PCT))
    d_cut = float(np.percentile(d, D_PCT))
    keep = (maxp >= p_cut) & (d <= d_cut)                                    # Filter 1: confidence & manifold
    keep &= ~((pred == 1) & (cen < thr_c))                                   # Filter 2: veto Cendrawasih (redundan setelah Cell 9)
    keep &= ~((pred == 2) & ((kut < thr_k) | (maxp < KUT_CONF_FLOOR)))       # Filter 3: Kutilang shield
    return keep, pred, p_cut, d_cut

keep_te, pred_te, p_thresh, d_thresh = curate(
    test_p_anchor, test_d_perch, cen_test, kut_test, final_thr_cendra, final_thr_kutilang
)
pseudo_indices = np.where(keep_te)[0].astype(np.int64)
pseudo_targets = pred_te[keep_te].astype(np.int64)
dist_pseudo = {IDX_TO_CLASS[c]: int((pseudo_targets == c).sum()) for c in range(NUM_CLASSES)}

print("\nconsensus_inlierness_curation_locked:")
print(f"  Confidence cutoff (p{P_PCT:.0f})   : {p_thresh:.3f}")
print(f"  Manifold dist cutoff (p{D_PCT:.0f}): {d_thresh:.3f}")
print(f"  Curated pseudo-samples    : {len(pseudo_indices)}/{n_te} ({len(pseudo_indices) / n_te * 100:.1f}%)")
print(f"  Pseudo-label distribution : {dist_pseudo}")

# 5. Diagnostik kemurnian pada OOF (aturan sama; ambang Kutilang per fold; hanya diagnostik)
thr_c_oof = np.array([fold_thrs[str(int(f))]["chosen_cendra"] for f in folds])
thr_k_oof = np.array([fold_thrs[str(int(f))]["chosen_kutilang"] for f in folds])
keep_oof, pred_oof, _, _ = curate(oof_p_reassigned, oof_d_perch, cen_oof, kut_oof, thr_c_oof, thr_k_oof)
purity = float((pred_oof[keep_oof] == y_target[keep_oof]).mean()) if keep_oof.any() else float("nan")
print(f"\noof_curation_diagnostic (aturan sama diterapkan pada OOF):")
print(f"  Lolos kurasi : {int(keep_oof.sum())}/{n_tr} | kemurnian pseudo-label = {purity:.4f}")
purity_cls = {}
for c in range(NUM_CLASSES):
    m = keep_oof & (pred_oof == c)
    purity_cls[IDX_TO_CLASS[c]] = {"n": int(m.sum()),
                                   "purity": float((y_target[m] == c).mean()) if m.any() else None}
    pv = purity_cls[IDX_TO_CLASS[c]]["purity"]
    print(f"  {IDX_TO_CLASS[c]:<12} n={int(m.sum()):<4} purity={'n/a' if pv is None else f'{pv:.4f}'}")

# 6. Simpan DULU, baru peringatan
np.save(WORK_DIR / "curated_pseudo_indices.npy", pseudo_indices)
np.save(WORK_DIR / "curated_pseudo_targets.npy", pseudo_targets)

meta_c10 = {
    "k_neighbors": K_NEIGHBORS,
    "oof_mean_dist": float(oof_d_perch.mean()),
    "test_mean_dist": float(test_d_perch.mean()),
    "p_thresh": p_thresh,
    "d_thresh": d_thresh,
    "p_percentile": P_PCT,
    "d_percentile": D_PCT,
    "n_curated": int(len(pseudo_indices)),
    "pseudo_distribution": dist_pseudo,
    "oof_curation_purity": purity,
    "oof_curation_purity_per_class": purity_cls,
    "note": "ambang persentil dihitung dari distribusi test (transduktif); skor energi tidak dipakai",
}
with open(WORK_DIR / "curated_inlier_meta.json", "w") as fh:
    json.dump(meta_c10, fh, indent=2)

if not (100 <= len(pseudo_indices) <= 180):
    print(f"[warn] Jumlah pseudo-samples {len(pseudo_indices)} di luar rentang acuan [100, 180]")
for name, cnt in dist_pseudo.items():
    if cnt < 10:
        print(f"[warn] Pseudo-label {name} hanya {cnt} sampel")

print("artifacts_saved: curated_pseudo_indices.npy, curated_pseudo_targets.npy, oof_d_perch.npy, test_d_perch.npy, test_nll_maxp.npy")
print(f"cell10_completed | elapsed={time.time() - t0_c10:.2f}s")

computing_knn_distances (k=5, in-fold)...
  oof_manifold_distance  : mean=0.3315 | p50=0.3188 | p90=0.4837
  test_manifold_distance : mean=0.5110 | p50=0.5254 | p90=0.7054

consensus_inlierness_curation_locked:
  Confidence cutoff (p75)   : 0.740
  Manifold dist cutoff (p40): 0.479
  Curated pseudo-samples    : 99/814 (12.2%)
  Pseudo-label distribution : {'Bondol Jawa': 43, 'Cendrawasih': 10, 'Kutilang': 46}

oof_curation_diagnostic (aturan sama diterapkan pada OOF):
  Lolos kurasi : 169/1826 | kemurnian pseudo-label = 0.9941
  Bondol Jawa  n=131  purity=0.9924
  Cendrawasih  n=3    purity=1.0000
  Kutilang     n=35   purity=1.0000
[warn] Jumlah pseudo-samples 99 di luar rentang acuan [100, 180]
artifacts_saved: curated_pseudo_indices.npy, curated_pseudo_targets.npy, oof_d_perch.npy, test_d_perch.npy, test_nll_maxp.npy
cell10_completed | elapsed=2.30s


In [11]:
# Cell 11 - In-Fold Augmented Dataset Construction (Train 1543-d + Pseudo-Test per Fold)
import json
import time
from pathlib import Path
import numpy as np
import pandas as pd

t0_c11 = time.time()
NUM_CLASSES = 3
CLASS_TO_IDX = {"Bondol Jawa": 0, "Cendrawasih": 1, "Kutilang": 2}
IDX_TO_CLASS = {v: k for k, v in CLASS_TO_IDX.items()}

# 1. Muat Fitur Hybrid 1543-d & Pseudo-Labels Terkurasi Cell 10
train_hybrid = np.load(WORK_DIR / "train_hybrid_1543.npy")   # (1826, 3, 1543)
test_hybrid = np.load(WORK_DIR / "test_hybrid_1543.npy")     # (814, 3, 1543)

pseudo_indices = np.load(WORK_DIR / "curated_pseudo_indices.npy")  # (99,)
pseudo_targets = np.load(WORK_DIR / "curated_pseudo_targets.npy")  # (99,)

n_tr, n_te = len(train_df), len(sub_df)
n_pseudo = len(pseudo_indices)
assert train_hybrid.shape == (n_tr, 3, 1543) and test_hybrid.shape == (n_te, 3, 1543)
assert len(pseudo_indices) == len(pseudo_targets)

# Ekstrak tensor pseudo test
pseudo_x = test_hybrid[pseudo_indices]  # (99, 3, 1543)
pseudo_y = pseudo_targets               # (99,)

y_target = train_df["target"].values
folds = train_df["fold"].values
fold_ids = np.unique(folds)
N_FOLDS = len(fold_ids)

# 2. Verifikasi Partisi Lipatan & Re-balancing Kelas per Fold
augmented_fold_info = {}

print(f"building_in_fold_augmented_partitions (N_pseudo={n_pseudo})...")
for f in fold_ids:
    tr_mask = folds != f
    val_mask = folds == f

    n_tr_orig = int(tr_mask.sum())
    n_val_orig = int(val_mask.sum())

    # Distribusi kelas data latih asli vs augmented
    orig_counts = [int((y_target[tr_mask] == c).sum()) for c in range(NUM_CLASSES)]
    pseudo_counts = [int((pseudo_y == c).sum()) for c in range(NUM_CLASSES)]
    aug_counts = [orig_counts[c] + pseudo_counts[c] for c in range(NUM_CLASSES)]

    # Cendrawasih di data latih bertambah 10 sampel inlier
    cendra_boost_pct = (pseudo_counts[1] / max(orig_counts[1], 1)) * 100

    augmented_fold_info[int(f)] = {
        "n_train_orig": n_tr_orig,
        "n_train_aug": n_tr_orig + n_pseudo,
        "n_val_pure_id": n_val_orig,
        "class_counts_orig": {IDX_TO_CLASS[c]: orig_counts[c] for c in range(NUM_CLASSES)},
        "class_counts_aug": {IDX_TO_CLASS[c]: aug_counts[c] for c in range(NUM_CLASSES)},
        "cendra_boost_samples": pseudo_counts[1],
        "cendra_boost_pct": round(cendra_boost_pct, 1),
    }

    print(
        f"  fold={f} | train_total={n_tr_orig + n_pseudo:<4} (orig={n_tr_orig} + pseudo={n_pseudo}) | "
        f"val_pure={n_val_orig:<3} | Cendra: {orig_counts[1]} -> {aug_counts[1]} (+{pseudo_counts[1]})"
    )

# 3. Validasi Keseluruhan: Validasi Wajib 100% Pure In-Distribution (Zero Pseudo di Val)
val_total_check = sum(v["n_val_pure_id"] for v in augmented_fold_info.values())
assert val_total_check == n_tr, f"Total sampel validasi {val_total_check} != {n_tr}"

# 4. Simpan Metadata Partisi Augmented
meta_c11 = {
    "n_pseudo_total": n_pseudo,
    "pseudo_indices": [int(i) for i in pseudo_indices],
    "pseudo_distribution": {IDX_TO_CLASS[c]: int((pseudo_y == c).sum()) for c in range(NUM_CLASSES)},
    "fold_partitions": augmented_fold_info,
}

with open(WORK_DIR / "augmented_in_fold_meta.json", "w") as fh:
    json.dump(meta_c11, fh, indent=2)

print(f"\nin_fold_augmented_ready | 4 folds configured | total_pseudo_injected={n_pseudo}")
print(f"saved_metadata -> {WORK_DIR / 'augmented_in_fold_meta.json'}")
print(f"cell11_completed | elapsed={time.time()-t0_c11:.2f}s")

building_in_fold_augmented_partitions (N_pseudo=99)...
  fold=0 | train_total=1422 (orig=1323 + pseudo=99) | val_pure=503 | Cendra: 48 -> 58 (+10)
  fold=1 | train_total=1463 (orig=1364 + pseudo=99) | val_pure=462 | Cendra: 54 -> 64 (+10)
  fold=2 | train_total=1503 (orig=1404 + pseudo=99) | val_pure=422 | Cendra: 67 -> 77 (+10)
  fold=3 | train_total=1486 (orig=1387 + pseudo=99) | val_pure=439 | Cendra: 80 -> 90 (+10)

in_fold_augmented_ready | 4 folds configured | total_pseudo_injected=99
saved_metadata -> /kaggle/working/augmented_in_fold_meta.json
cell11_completed | elapsed=0.02s


In [12]:
# Cell 12 - Retraining 4-Fold Bottleneck Cosine Heads (Self-Training, fitur dibangun on-the-fly per fold)
import json
import time
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix, f1_score, recall_score, roc_auc_score
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

t0_c12 = time.time()

SEED = 42
N_EPOCHS_HEAD = 40
BATCH_SIZE_HEAD = 32
EMB_DIM = 256
NUM_CLASSES = 3
PSEUDO_W = 1.0   # 1.0 = setara label asli; coba 0.5 sebagai varian

CLASS_TO_IDX = {"Bondol Jawa": 0, "Cendrawasih": 1, "Kutilang": 2}
IDX_TO_CLASS = {v: k for k, v in CLASS_TO_IDX.items()}

# 1. Muat data mentah, pseudo-label Cell 10, dan baseline Cell 7
with open(WORK_DIR / "perch_taxa_indices.json") as fh:
    taxa_meta = json.load(fh)
cendra_names = list(taxa_meta["cendra"].keys())
n_cand = len(cendra_names)

train_raw_sum = np.load(WORK_DIR / "train_raw_cand_summary.npy")
test_raw_sum = np.load(WORK_DIR / "test_raw_cand_summary.npy")
X_tr_emb = np.load(WORK_DIR / "perch_v2_train_emb.npy")
X_te_emb = np.load(WORK_DIR / "perch_v2_test_emb.npy")
oof_probs_base = np.load(WORK_DIR / "oof_p_supervised.npy")
test_probs_base = np.load(WORK_DIR / "test_p_supervised.npy")

pseudo_indices = np.load(WORK_DIR / "curated_pseudo_indices.npy")
pseudo_targets = np.load(WORK_DIR / "curated_pseudo_targets.npy").astype(np.int64)
n_pseudo = len(pseudo_indices)

y_target = train_df["target"].values
folds = train_df["fold"].values
fold_ids = np.unique(folds)
N_FOLDS = len(fold_ids)
n_tr, n_te = len(train_df), len(sub_df)
IN_FEATURES = X_tr_emb.shape[-1] + 7

assert X_tr_emb.shape[0] == n_tr and X_te_emb.shape[0] == n_te, "Fitur tidak sejajar dengan metadata"
assert oof_probs_base.shape == (n_tr, NUM_CLASSES) and test_probs_base.shape == (n_te, NUM_CLASSES)
assert len(pseudo_targets) == n_pseudo and len(np.unique(pseudo_indices)) == n_pseudo, "Pseudo-label tidak valid"
assert n_pseudo == 0 or (pseudo_indices.min() >= 0 and pseudo_indices.max() < n_te), "Indeks pseudo di luar rentang"
assert set(np.unique(pseudo_targets)) <= set(range(NUM_CLASSES)), "Label pseudo tidak valid"

# 2. Fungsi fitur (identik dengan Cell 7)
COL_B, COL_K = 0, 1
cendra_cols = list(range(2, 2 + n_cand))
COL_MAX_NT, COL_TOP5_NT, COL_ENERGY = 2 + n_cand, 3 + n_cand, 4 + n_cand
y_cendra_all = (y_target == 1).astype(int)
cand_max = np.max(train_raw_sum[:, :, 2:2 + n_cand], axis=1)

def safe_auc(y_true, scores):
    return float(roc_auc_score(y_true, scores)) if len(np.unique(y_true)) == 2 else np.nan

def pick_cendra(tr_mask):
    aucs = np.array([safe_auc(y_cendra_all[tr_mask], cand_max[tr_mask, j]) for j in range(n_cand)])
    return int(np.nanargmax(aucs))

def build_summary_7d(raw, c_col):
    z_b, z_k = raw[:, :, COL_B:COL_B + 1], raw[:, :, COL_K:COL_K + 1]
    z_c = raw[:, :, c_col:c_col + 1]
    max_nt, top5_nt = raw[:, :, COL_MAX_NT:COL_MAX_NT + 1], raw[:, :, COL_TOP5_NT:COL_TOP5_NT + 1]
    energy = raw[:, :, COL_ENERGY:COL_ENERGY + 1]
    margin_ood = max_nt - np.maximum(np.maximum(z_b, z_k), z_c)
    return np.concatenate([z_b, z_c, z_k, max_nt, top5_nt, margin_ood, energy], axis=2).astype(np.float32)

def make_hybrid(raw, emb, c_col, mu, sd):
    return np.concatenate([emb, (build_summary_7d(raw, c_col) - mu) / sd], axis=-1).astype(np.float32)

# 3. Arsitektur
class CosineLinear(nn.Module):
    def __init__(self, in_features, num_classes=3, init_tau=16.0):
        super().__init__()
        self.weight = nn.Parameter(torch.randn(num_classes, in_features))
        nn.init.xavier_uniform_(self.weight)
        self.tau = nn.Parameter(torch.tensor(init_tau, dtype=torch.float32))

    def forward(self, x):
        x_norm = F.normalize(x, p=2, dim=-1)
        w_norm = F.normalize(self.weight, p=2, dim=-1)
        return torch.clamp(self.tau, min=1.0, max=32.0) * F.linear(x_norm, w_norm)

class PerchBottleneckCosineHead(nn.Module):
    def __init__(self, in_features=IN_FEATURES, emb_dim=EMB_DIM, num_classes=NUM_CLASSES, dropout=0.2, init_tau=16.0):
        super().__init__()
        self.bottleneck = nn.Sequential(
            nn.Linear(in_features, emb_dim, bias=False),
            nn.LayerNorm(emb_dim),
            nn.SiLU(inplace=True),
            nn.Dropout(p=dropout),
        )
        self.classifier = CosineLinear(emb_dim, num_classes=num_classes, init_tau=init_tau)

    def forward(self, x_3w):
        B, W, D = x_3w.shape
        h = self.bottleneck(x_3w.reshape(B * W, D))
        h_norm = F.normalize(h, p=2, dim=-1)
        logits_3w = self.classifier(h_norm).view(B, W, -1)
        h_3w = h_norm.view(B, W, -1)
        conf_weights = F.softmax(logits_3w.max(dim=-1, keepdim=True)[0] * 2.0, dim=1)
        bag_logits = (logits_3w * conf_weights).sum(dim=1)
        bag_emb = F.normalize((h_3w * conf_weights).sum(dim=1), p=2, dim=-1)
        return bag_logits, bag_emb

# 4. Class-Balanced Focal Loss dengan bobot per sampel
def compute_effective_num_weights(class_counts, beta=0.99):
    counts = np.asarray(class_counts, dtype=np.float64)
    weights = (1.0 - beta) / (1.0 - np.power(beta, counts))
    weights = weights / np.sum(weights) * len(counts)
    return torch.tensor(weights, dtype=torch.float32)

class ClassBalancedFocalLoss(nn.Module):
    def __init__(self, class_counts, beta=0.99, gamma=2.0, smoothing=0.05):
        super().__init__()
        self.gamma = gamma
        self.smoothing = smoothing
        self.num_classes = len(class_counts)
        self.register_buffer("alpha", compute_effective_num_weights(class_counts, beta=beta))

    def forward(self, logits, targets, sample_w=None):
        log_probs = F.log_softmax(logits, dim=-1)
        probs = torch.exp(log_probs)
        with torch.no_grad():
            true_dist = torch.zeros_like(logits)
            true_dist.fill_(self.smoothing / (self.num_classes - 1))
            true_dist.scatter_(1, targets.unsqueeze(1), 1.0 - self.smoothing)
        focal_weight = torch.pow(1.0 - probs, self.gamma)
        loss = -torch.sum(self.alpha.unsqueeze(0) * focal_weight * true_dist * log_probs, dim=-1)
        if sample_w is None:
            return loss.mean()
        return (loss * sample_w).sum() / sample_w.sum()

# 5. Pelatihan 4-fold: partisi dibangun on-the-fly (seed sama dengan Cell 7)
oof_probs_st = np.zeros((n_tr, NUM_CLASSES), dtype=np.float32)
oof_emb256_st = np.zeros((n_tr, EMB_DIM), dtype=np.float32)
test_probs_st = np.zeros((n_te, NUM_CLASSES), dtype=np.float32)
test_emb256_st_folds = np.zeros((N_FOLDS, n_te, EMB_DIM), dtype=np.float32)
fold_f1s_st, pseudo_keep_rates = [], []
fold_species = {}

print(f"retraining_self_trained_heads ({N_EPOCHS_HEAD} epochs per fold, n_pseudo={n_pseudo}, pseudo_w={PSEUDO_W})...")
for k, fold in enumerate(fold_ids):
    t_f = time.time()
    torch.manual_seed(SEED + int(fold))
    np.random.seed(SEED + int(fold))

    tr_mask = folds != fold
    tr_idx = np.where(tr_mask)[0]
    val_idx = np.where(~tr_mask)[0]
    assert set(tr_idx).isdisjoint(set(val_idx))

    # seleksi taksa + z-score HANYA dari baris train fold ini
    j = pick_cendra(tr_mask)
    c_col = cendra_cols[j]
    fold_species[int(fold)] = cendra_names[j]
    s7_tr = build_summary_7d(train_raw_sum[tr_idx], c_col)
    mu = s7_tr.mean(axis=(0, 1), keepdims=True)
    sd = s7_tr.std(axis=(0, 1), keepdims=True) + 1e-6

    x_tr = make_hybrid(train_raw_sum[tr_idx], X_tr_emb[tr_idx], c_col, mu, sd)
    x_ps = make_hybrid(test_raw_sum[pseudo_indices], X_te_emb[pseudo_indices], c_col, mu, sd)
    x_va = make_hybrid(train_raw_sum[val_idx], X_tr_emb[val_idx], c_col, mu, sd)
    x_te = make_hybrid(test_raw_sum, X_te_emb, c_col, mu, sd)

    x_aug = np.concatenate([x_tr, x_ps], axis=0)
    y_aug = np.concatenate([y_target[tr_idx], pseudo_targets]).astype(np.int64)
    is_pseudo = np.concatenate([np.zeros(len(tr_idx), bool), np.ones(n_pseudo, bool)])
    y_va = y_target[val_idx]
    for arr in (x_aug, x_va, x_te):
        assert np.isfinite(arr).all()
    assert x_aug.shape[1:] == (3, IN_FEATURES)

    x_va_t = torch.from_numpy(x_va).to(DEVICE)
    x_te_t = torch.from_numpy(x_te).to(DEVICE)

    sample_w = np.where(is_pseudo, PSEUDO_W, 1.0).astype(np.float32)
    ds = TensorDataset(torch.from_numpy(x_aug), torch.from_numpy(y_aug), torch.from_numpy(sample_w))
    gen = torch.Generator().manual_seed(SEED + int(fold))
    dl_tr = DataLoader(ds, batch_size=BATCH_SIZE_HEAD, shuffle=True, generator=gen)

    model = PerchBottleneckCosineHead().to(DEVICE)
    decay, no_decay = [], []
    for _, p in model.named_parameters():
        (no_decay if p.ndim <= 1 else decay).append(p)
    optimizer = torch.optim.AdamW(
        [{"params": decay, "weight_decay": 1e-4}, {"params": no_decay, "weight_decay": 0.0}], lr=1e-3,
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=N_EPOCHS_HEAD, eta_min=1e-5)

    aug_counts = [int(np.sum(y_aug == c)) for c in range(NUM_CLASSES)]
    criterion = ClassBalancedFocalLoss(aug_counts, beta=0.99, gamma=2.0).to(DEVICE)

    for ep in range(1, N_EPOCHS_HEAD + 1):
        model.train()
        for bx, by, bw in dl_tr:
            bx, by, bw = bx.to(DEVICE), by.to(DEVICE), bw.to(DEVICE)
            optimizer.zero_grad()
            logits, _ = model(bx)
            criterion(logits, by, bw).backward()
            optimizer.step()
        scheduler.step()

    model.eval()
    with torch.no_grad():
        v_logits, v_emb = model(x_va_t)
        v_probs = F.softmax(v_logits, dim=-1).cpu().numpy()
        v_emb = v_emb.cpu().numpy()
        t_logits, t_emb = model(x_te_t)
        t_probs = F.softmax(t_logits, dim=-1).cpu().numpy()
        t_emb = t_emb.cpu().numpy()

    v_pred = np.argmax(v_probs, axis=1)
    final_f1 = float(f1_score(y_va, v_pred, average="macro"))
    cendra_f1 = float(f1_score(y_va == 1, v_pred == 1, zero_division=0))
    fold_f1s_st.append(final_f1)

    keep_rate = float((t_probs[pseudo_indices].argmax(axis=1) == pseudo_targets).mean()) if n_pseudo else float("nan")
    pseudo_keep_rates.append(keep_rate)

    oof_probs_st[val_idx] = v_probs
    oof_emb256_st[val_idx] = v_emb
    test_probs_st += t_probs / N_FOLDS
    test_emb256_st_folds[k] = t_emb
    torch.save(model.state_dict(), WORK_DIR / f"head_selftrained_fold{int(fold)}.pt")

    print(f"  fold={k + 1}/{N_FOLDS} | cendra_sp='{cendra_names[j]}' | train={len(x_aug):<4} "
          f"(orig={len(tr_idx)} + pseudo={n_pseudo}) | val={len(val_idx):<4} | macro_f1={final_f1:.4f} | "
          f"cendra_f1={cendra_f1:.4f} | pseudo_keep={keep_rate:.3f} | {time.time() - t_f:.1f}s")

    del model, optimizer, x_va_t, x_te_t, ds, x_aug, x_tr, x_ps

# 6. Evaluasi OOF dan perbandingan langsung dengan baseline Cell 7
oof_preds_st = np.argmax(oof_probs_st, axis=1)
oof_preds_base = np.argmax(oof_probs_base, axis=1)
f1_st = f1_score(y_target, oof_preds_st, average="macro")
f1_base = f1_score(y_target, oof_preds_base, average="macro")
pc_st, pc_base = f1_score(y_target, oof_preds_st, average=None), f1_score(y_target, oof_preds_base, average=None)
rec_st, rec_base = recall_score(y_target, oof_preds_st, average=None), recall_score(y_target, oof_preds_base, average=None)
cm_st = confusion_matrix(y_target, oof_preds_st, labels=[0, 1, 2])

print("\nself_training_completed | OOF murni (validasi hanya data train asli)")
print(f"  Macro F1 baseline (Cell 7) : {f1_base:.4f}")
print(f"  Macro F1 self-trained      : {f1_st:.4f} (delta={f1_st - f1_base:+.4f})")
print(f"  Mean per-fold F1           : {np.mean(fold_f1s_st):.4f} +/- {np.std(fold_f1s_st):.4f}")
print(f"Confusion Matrix (row=true, col=pred):\n{cm_st}")
for idx, name in IDX_TO_CLASS.items():
    print(f"  {name:<12} F1 {pc_base[idx]:.4f} -> {pc_st[idx]:.4f} ({pc_st[idx] - pc_base[idx]:+.4f}) | "
          f"recall {rec_base[idx]:.3f} -> {rec_st[idx]:.3f}")
print(f"  pseudo_keep rata-rata antar fold: {np.nanmean(pseudo_keep_rates):.3f} (mendekati 1 = model hanya meniru pseudo-label)")
changed = int((np.argmax(test_probs_st, axis=1) != np.argmax(test_probs_base, axis=1)).sum())
print(f"  Prediksi test yang berubah vs supervised: {changed}/{n_te}")
print(f"  Spesies Cendrawasih pemenang per fold: {list(fold_species.values())}")

# 7. Simpan artefak
np.save(WORK_DIR / "oof_p_selftrained.npy", oof_probs_st)
np.save(WORK_DIR / "test_p_selftrained.npy", test_probs_st)
np.save(WORK_DIR / "oof_emb_selftrained_256d.npy", oof_emb256_st)
np.save(WORK_DIR / "test_emb_selftrained_256d_per_fold.npy", test_emb256_st_folds)
np.save(WORK_DIR / "test_emb_selftrained_256d.npy", test_emb256_st_folds.mean(axis=0))  # kompatibilitas; ruang antar fold tidak selaras
with open(WORK_DIR / "selftrained_fold_species.json", "w") as fh:
    json.dump(fold_species, fh, indent=2)

print("artifacts_saved: oof_p_selftrained.npy, test_p_selftrained.npy, test_emb_selftrained_256d_per_fold.npy")
print(f"cell12_completed | elapsed={time.time() - t0_c12:.1f}s")

retraining_self_trained_heads (40 epochs per fold, n_pseudo=99, pseudo_w=1.0)...
  fold=1/4 | cendra_sp='Cicinnurus regius' | train=1422 (orig=1323 + pseudo=99) | val=503  | macro_f1=0.9719 | cendra_f1=0.9231 | pseudo_keep=1.000 | 9.4s
  fold=2/4 | cendra_sp='Cicinnurus regius' | train=1463 (orig=1364 + pseudo=99) | val=462  | macro_f1=1.0000 | cendra_f1=1.0000 | pseudo_keep=1.000 | 9.7s
  fold=3/4 | cendra_sp='Paradisaea rubra' | train=1503 (orig=1404 + pseudo=99) | val=422  | macro_f1=0.8342 | cendra_f1=0.5385 | pseudo_keep=1.000 | 9.9s
  fold=4/4 | cendra_sp='Paradisaea rubra' | train=1486 (orig=1387 + pseudo=99) | val=439  | macro_f1=0.6612 | cendra_f1=0.0000 | pseudo_keep=1.000 | 9.8s

self_training_completed | OOF murni (validasi hanya data train asli)
  Macro F1 baseline (Cell 7) : 0.9119
  Macro F1 self-trained      : 0.9421 (delta=+0.0302)
  Mean per-fold F1           : 0.8668 +/- 0.1343
Confusion Matrix (row=true, col=pred):
[[1114    8    2]
 [  15   66    2]
 [   0    0  61

In [13]:
# Cell 13 - Self-Trained Forensic Reassignment & Adapted Test Matrix
import json
import time
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix, f1_score, precision_score, recall_score

t0_c13 = time.time()
NUM_CLASSES = 3
KUT_CONF_FLOOR = 0.60

CLASS_TO_IDX = {"Bondol Jawa": 0, "Cendrawasih": 1, "Kutilang": 2}
IDX_TO_CLASS = {v: k for k, v in CLASS_TO_IDX.items()}

# 1. Muat prediksi Cell 12, ambang Cell 8, dan BirdNET
oof_p_st = np.load(WORK_DIR / "oof_p_selftrained.npy")
test_p_st = np.load(WORK_DIR / "test_p_selftrained.npy")

with open(WORK_DIR / "nested_forensic_thresholds.json") as fh:
    thr_meta = json.load(fh)
fold_thrs = thr_meta["fold_thresholds"]
final_thr_cendra = thr_meta["final_thr_cendra"]
final_thr_kutilang = thr_meta["final_thr_kutilang"]

train_bnet = np.load(WORK_DIR / "birdnet_train_summary.npy")
test_bnet = np.load(WORK_DIR / "birdnet_test_summary.npy")

n_tr, n_te = len(train_df), len(sub_df)
assert oof_p_st.shape == (n_tr, NUM_CLASSES) and test_p_st.shape == (n_te, NUM_CLASSES), "Shape prediksi tidak sejajar"
assert train_bnet.shape[0] == n_tr and test_bnet.shape[0] == n_te, "Shape BirdNET tidak sejajar"

cen_oof, kut_oof = train_bnet[:, :, 2].max(axis=1), train_bnet[:, :, 1].max(axis=1)
cen_test, kut_test = test_bnet[:, :, 2].max(axis=1), test_bnet[:, :, 1].max(axis=1)

y_true = train_df["target"].values
folds = train_df["fold"].values
assert {str(int(f)) for f in np.unique(folds)} <= set(fold_thrs), "Kunci fold tidak ada di JSON"

# 2. Fungsi reassignment (sama dengan Cell 9)
def reassign(p, score, thr):
    pred_raw = np.argmax(p, axis=1)
    second = np.argsort(p, axis=1)[:, -2]
    mask = (pred_raw == 1) & (score < thr)
    corr = pred_raw.copy()
    corr[mask] = second[mask]
    p2 = p.copy()
    idx = np.where(mask)[0]
    p2[idx, second[idx]] += p2[idx, 1]
    p2[idx, 1] = 0.0
    p2 /= p2.sum(axis=1, keepdims=True)
    return pred_raw, corr, p2, mask

# 3. OOF: reassignment dengan ambang per fold
thr_c_oof = np.array([fold_thrs[str(int(f))]["chosen_cendra"] for f in folds])
thr_k_oof = np.array([fold_thrs[str(int(f))]["chosen_kutilang"] for f in folds])
pred_raw_st, corr_pred_st, oof_p_st_reassigned, mask_oof = reassign(oof_p_st, cen_oof, thr_c_oof)
oof_reassigned_st = int(mask_oof.sum())

f1_raw_st = float(f1_score(y_true, pred_raw_st, average="macro"))
f1_corr_st = float(f1_score(y_true, corr_pred_st, average="macro"))
f1_pc_corr_st = f1_score(y_true, corr_pred_st, average=None)
cm_raw_st = confusion_matrix(y_true, pred_raw_st, labels=[0, 1, 2])
cm_corr_st = confusion_matrix(y_true, corr_pred_st, labels=[0, 1, 2])

tp_lost = int((mask_oof & (y_true == 1)).sum())
fp_removed = int((mask_oof & (y_true != 1)).sum())
fp_fixed = int((mask_oof & (y_true != 1) & (corr_pred_st == y_true)).sum())
rec_raw = float(recall_score(y_true == 1, pred_raw_st == 1, zero_division=0))
rec_corr = float(recall_score(y_true == 1, corr_pred_st == 1, zero_division=0))
prec_raw = float(precision_score(y_true == 1, pred_raw_st == 1, zero_division=0))
prec_corr = float(precision_score(y_true == 1, corr_pred_st == 1, zero_division=0))

print("self_trained_oof_reassignment_audit:")
print(f"  Dipindahkan dari Cendrawasih  : {oof_reassigned_st} (FP dibuang={fp_removed}, benar={fp_fixed} | TP hilang={tp_lost})")
print(f"  Cendra recall / precision     : {rec_raw:.3f} / {prec_raw:.3f}  ->  {rec_corr:.3f} / {prec_corr:.3f}")
print(f"  Macro F1 sebelum              : {f1_raw_st:.4f}")
print(f"  Macro F1 sesudah              : {f1_corr_st:.4f} (delta={f1_corr_st - f1_raw_st:+.4f})")
print(f"  Per-class F1 sesudah          : Bondol={f1_pc_corr_st[0]:.4f} | Cendra={f1_pc_corr_st[1]:.4f} | Kutilang={f1_pc_corr_st[2]:.4f}")
print(f"Confusion Matrix sebelum (row=true, col=pred):\n{cm_raw_st}")
print(f"Confusion Matrix sesudah:\n{cm_corr_st}")

# Perbandingan dengan Cell 9 (supervised), jika ada
c9_path = WORK_DIR / "supervised_reassignment_meta.json"
f1_c9 = None
if c9_path.exists():
    with open(c9_path) as fh:
        c9 = json.load(fh)
    f1_c9 = c9.get("oof_macro_f1_reassigned")
    if f1_c9 is not None:
        print(f"  Macro F1 supervised+reassign (Cell 9): {f1_c9:.4f} | self-trained+reassign: {f1_corr_st:.4f} (delta={f1_corr_st - f1_c9:+.4f})")

# 4. Marker Trucuk Shield: OOF (diagnostik) dan test
def trucuk_flag(p, kut, thr_k):
    return (np.argmax(p, axis=1) == 2) & (p[:, 2] < KUT_CONF_FLOOR) & (kut < thr_k)

flag_oof = trucuk_flag(oof_p_st, kut_oof, thr_k_oof)
n_flag_oof = int(flag_oof.sum())
flag_true_kut = float((y_true[flag_oof] == 2).mean()) if n_flag_oof else float("nan")
print(f"\ntrucuk_shield_oof_diagnostic:")
print(f"  OOF ditandai: {n_flag_oof}/{n_tr} | dari itu Kutilang asli: {flag_true_kut:.3f}")
print(f"  (Jika mendekati 1, marker ini menandai Kutilang asli, bukan Alien Bulbul)")

pred_raw_test_st, corr_pred_test_st, test_p_st_adapted, mask_test = reassign(test_p_st, cen_test, final_thr_cendra)
test_reassigned_st_count = int(mask_test.sum())
trucuk_mask_test = trucuk_flag(test_p_st, kut_test, final_thr_kutilang) & ~mask_test

print("\ntest_self_trained_forensics_applied:")
print(f"  Test Cendrawasih calls raw         : {int((pred_raw_test_st == 1).sum())}")
print(f"  Test Cendrawasih calls reassigned  : {test_reassigned_st_count}")
print(f"  Test Cendrawasih calls clean       : {int((corr_pred_test_st == 1).sum())}")
print(f"  Trucuk shield ditandai (marker)    : {int(trucuk_mask_test.sum())} samples")
print(f"  Test distribution raw              : {pd.Series([IDX_TO_CLASS[c] for c in pred_raw_test_st]).value_counts().to_dict()}")
print(f"  Test distribution reassigned       : {pd.Series([IDX_TO_CLASS[c] for c in corr_pred_test_st]).value_counts().to_dict()}")

# 5. Simpan
np.save(WORK_DIR / "oof_p_selftrained_reassigned.npy", oof_p_st_reassigned)
np.save(WORK_DIR / "test_p_selftrained_adapted.npy", test_p_st_adapted)
np.save(WORK_DIR / "trucuk_mask_test.npy", trucuk_mask_test)

meta_c13 = {
    "oof_macro_f1_st_raw": f1_raw_st,
    "oof_macro_f1_st_reassigned": f1_corr_st,
    "oof_macro_f1_supervised_reassigned_cell9": f1_c9,
    "oof_reassigned_st_samples": oof_reassigned_st,
    "oof_fp_removed": fp_removed,
    "oof_fp_fixed": fp_fixed,
    "oof_tp_lost": tp_lost,
    "oof_cendra_recall": [rec_raw, rec_corr],
    "oof_cendra_precision": [prec_raw, prec_corr],
    "trucuk_oof_flagged": n_flag_oof,
    "trucuk_oof_flagged_true_kutilang_rate": None if n_flag_oof == 0 else flag_true_kut,
    "test_cendra_st_raw": int((pred_raw_test_st == 1).sum()),
    "test_cendra_st_reassigned": test_reassigned_st_count,
    "test_cendra_st_clean": int((corr_pred_test_st == 1).sum()),
    "trucuk_shielded_test": int(trucuk_mask_test.sum()),
    "cm_raw_st": cm_raw_st.tolist(),
    "cm_reassigned_st": cm_corr_st.tolist(),
}
with open(WORK_DIR / "selftrained_reassignment_meta.json", "w") as fh:
    json.dump(meta_c13, fh, indent=2)

print(f"\ncell13_completed | final_self_trained_oof_f1={f1_corr_st:.4f} | elapsed={time.time() - t0_c13:.2f}s")

self_trained_oof_reassignment_audit:
  Dipindahkan dari Cendrawasih  : 7 (FP dibuang=6, benar=6 | TP hilang=1)
  Cendra recall / precision     : 0.795 / 0.892  ->  0.783 / 0.970
  Macro F1 sebelum              : 0.9421
  Macro F1 sesudah              : 0.9515 (delta=+0.0094)
  Per-class F1 sesudah          : Bondol=0.9912 | Cendra=0.8667 | Kutilang=0.9968
Confusion Matrix sebelum (row=true, col=pred):
[[1114    8    2]
 [  15   66    2]
 [   0    0  619]]
Confusion Matrix sesudah:
[[1120    2    2]
 [  16   65    2]
 [   0    0  619]]
  Macro F1 supervised+reassign (Cell 9): 0.9383 | self-trained+reassign: 0.9515 (delta=+0.0133)

trucuk_shield_oof_diagnostic:
  OOF ditandai: 4/1826 | dari itu Kutilang asli: 0.000
  (Jika mendekati 1, marker ini menandai Kutilang asli, bukan Alien Bulbul)

test_self_trained_forensics_applied:
  Test Cendrawasih calls raw         : 116
  Test Cendrawasih calls reassigned  : 19
  Test Cendrawasih calls clean       : 97
  Trucuk shield ditandai (marker)   

In [14]:
# Cell 14 - Dual Forensic Taxonomic Shield (Supervised Anchor & Self-Trained Matrices)
import json
import time
from pathlib import Path
import numpy as np
import pandas as pd

t0_c14 = time.time()
NUM_CLASSES = 3
KUT_CONF_FLOOR = 0.60

CLASS_TO_IDX = {"Bondol Jawa": 0, "Cendrawasih": 1, "Kutilang": 2}
IDX_TO_CLASS = {v: k for k, v in CLASS_TO_IDX.items()}

# 1. Muat matriks test (sudah di-reassign) dan ambang final
p_sup_anchor = np.load(WORK_DIR / "test_p_supervised_anchor.npy")     # Cell 9
p_st_adapted = np.load(WORK_DIR / "test_p_selftrained_adapted.npy")   # Cell 13
trucuk_c13 = np.load(WORK_DIR / "trucuk_mask_test.npy")               # Cell 13 (definisi lama, hanya untuk pembanding)

with open(WORK_DIR / "nested_forensic_thresholds.json") as fh:
    thr_meta = json.load(fh)
final_thr_kutilang = thr_meta["final_thr_kutilang"]

test_bnet = np.load(WORK_DIR / "birdnet_test_summary.npy")
kut_test = test_bnet[:, :, 1].max(axis=1)
n_te = len(sub_df)

assert p_sup_anchor.shape == (n_te, NUM_CLASSES) and p_st_adapted.shape == (n_te, NUM_CLASSES), "Shape tidak sejajar"
assert trucuk_c13.shape == (n_te,) and kut_test.shape == (n_te,)

# 2. Satu fungsi mask untuk kedua jalur (matriks yang sama-sama sudah di-reassign)
def trucuk_flag(p, kut, thr_k):
    return (np.argmax(p, axis=1) == 2) & (p[:, 2] < KUT_CONF_FLOOR) & (kut < thr_k)

pred_sup = np.argmax(p_sup_anchor, axis=1)
pred_st = np.argmax(p_st_adapted, axis=1)
trucuk_mask_sup = trucuk_flag(p_sup_anchor, kut_test, final_thr_kutilang)
trucuk_mask_st = trucuk_flag(p_st_adapted, kut_test, final_thr_kutilang)

n_sup, n_st = int(trucuk_mask_sup.sum()), int(trucuk_mask_st.sum())
inter = int((trucuk_mask_sup & trucuk_mask_st).sum())
union = int((trucuk_mask_sup | trucuk_mask_st).sum())
jaccard = inter / union if union else float("nan")
diff_c13 = int((trucuk_mask_st != trucuk_c13).sum())

# 3. Perbandingan dua jalur
dist_sup = pd.Series([IDX_TO_CLASS[c] for c in pred_sup]).value_counts().to_dict()
dist_st = pd.Series([IDX_TO_CLASS[c] for c in pred_st]).value_counts().to_dict()
disagree_idx = np.where(pred_sup != pred_st)[0]
agreement = 1.0 - len(disagree_idx) / n_te
xtab = pd.crosstab(pd.Series([IDX_TO_CLASS[c] for c in pred_sup], name="supervised"),
                   pd.Series([IDX_TO_CLASS[c] for c in pred_st], name="self-trained"))

print("dual_forensic_shield_summary:")
print(f"  Supervised  | Cendrawasih={dist_sup.get('Cendrawasih', 0)} | Trucuk ditandai={n_sup}")
print(f"  Self-Trained| Cendrawasih={dist_st.get('Cendrawasih', 0)} | Trucuk ditandai={n_st}")
print(f"  Overlap Trucuk: {inter} | dari sup={inter / max(n_sup, 1):.1%} | dari st={inter / max(n_st, 1):.1%} | Jaccard={jaccard:.3f}")
print(f"  Mask self-trained berbeda dari Cell 13 (definisi lama): {diff_c13} sampel")
print(f"  Supervised distribution   : {dist_sup}")
print(f"  Self-trained distribution : {dist_st}")
print(f"  Kesepakatan prediksi antar jalur: {agreement:.4f} ({len(disagree_idx)} berbeda dari {n_te})")
print(f"Crosstab (baris=supervised, kolom=self-trained):\n{xtab}")

# 4. Simpan
np.save(WORK_DIR / "trucuk_mask_supervised.npy", trucuk_mask_sup)
np.save(WORK_DIR / "trucuk_mask_selftrained.npy", trucuk_mask_st)
np.save(WORK_DIR / "branch_disagreement_idx.npy", disagree_idx.astype(np.int64))

meta_c14 = {
    "supervised_cendra_clean": int(dist_sup.get("Cendrawasih", 0)),
    "selftrained_cendra_clean": int(dist_st.get("Cendrawasih", 0)),
    "supervised_trucuk_flagged": n_sup,
    "selftrained_trucuk_flagged": n_st,
    "trucuk_overlap": inter,
    "trucuk_jaccard": jaccard,
    "selftrained_mask_diff_vs_cell13": diff_c13,
    "branch_agreement": float(agreement),
    "n_branch_disagreement": int(len(disagree_idx)),
    "dist_supervised": {k: int(v) for k, v in dist_sup.items()},
    "dist_selftrained": {k: int(v) for k, v in dist_st.items()},
    "note": "mask Trucuk hanya marker (tidak mengubah prediksi); kedua jalur tidak independen karena pseudo-label berasal dari jalur supervised",
}
with open(WORK_DIR / "dual_forensics_meta.json", "w") as fh:
    json.dump(meta_c14, fh, indent=2)

print("\nartifacts_saved: trucuk_mask_supervised.npy, trucuk_mask_selftrained.npy, branch_disagreement_idx.npy, dual_forensics_meta.json")
print(f"cell14_completed | elapsed={time.time() - t0_c14:.2f}s")

dual_forensic_shield_summary:
  Supervised  | Cendrawasih=126 | Trucuk ditandai=56
  Self-Trained| Cendrawasih=97 | Trucuk ditandai=52
  Overlap Trucuk: 39 | dari sup=69.6% | dari st=75.0% | Jaccard=0.565
  Mask self-trained berbeda dari Cell 13 (definisi lama): 0 sampel
  Supervised distribution   : {'Bondol Jawa': 429, 'Kutilang': 259, 'Cendrawasih': 126}
  Self-trained distribution : {'Bondol Jawa': 480, 'Kutilang': 237, 'Cendrawasih': 97}
  Kesepakatan prediksi antar jalur: 0.9152 (69 berbeda dari 814)
Crosstab (baris=supervised, kolom=self-trained):
self-trained  Bondol Jawa  Cendrawasih  Kutilang
supervised                                      
Bondol Jawa           421            2         6
Cendrawasih            31           94         1
Kutilang               28            1       230

artifacts_saved: trucuk_mask_supervised.npy, trucuk_mask_selftrained.npy, branch_disagreement_idx.npy, dual_forensics_meta.json
cell14_completed | elapsed=0.08s


In [15]:
# Cell 15 - Manifold Unknown Quantile Locking (laju Unknown = hyperparameter, dipilih dari umpan balik submission)
import json
import time
from pathlib import Path
import numpy as np

t0_c15 = time.time()

# 1. Muat jarak manifold & mask Trucuk
oof_d_perch = np.load(WORK_DIR / "oof_d_perch.npy")
test_d_perch = np.load(WORK_DIR / "test_d_perch.npy")
trucuk_mask_sup = np.load(WORK_DIR / "trucuk_mask_supervised.npy")
trucuk_mask_st = np.load(WORK_DIR / "trucuk_mask_selftrained.npy")

n_te = len(test_d_perch)
assert len(sub_df) == n_te and len(trucuk_mask_sup) == n_te and len(trucuk_mask_st) == n_te, "Shape tidak sejajar"
assert np.isfinite(test_d_perch).all() and np.isfinite(oof_d_perch).all()

# 2. Laju Unknown (hyperparameter; bukan hasil validasi OOF)
RATE_STANDARD = 0.4349
RATE_PROBE = 0.4152
TARGET_STANDARD_COUNT = int(round(RATE_STANDARD * n_te))   # 354 untuk n_te=814
TARGET_PROBE_COUNT = int(round(RATE_PROBE * n_te))         # 338 untuk n_te=814

def calibrate_tau(test_d, trucuk_mask, target_count):
    """Unknown = Trucuk-flag ATAU jarak terbesar; jumlah total persis target_count (berbasis ranking)."""
    cand = np.where(~trucuk_mask)[0]
    needed = target_count - int(trucuk_mask.sum())
    assert 0 < needed <= len(cand), (
        f"Trucuk={int(trucuk_mask.sum())} vs target={target_count}: needed={needed}, kandidat={len(cand)}")
    order = np.argsort(-test_d[cand], kind="stable")[:needed]
    is_unk = trucuk_mask.copy()
    is_unk[cand[order]] = True
    tau = float(test_d[cand[order]].min())                  # ambang inklusif: d >= tau
    oof_reject = float((oof_d_perch >= tau).mean())         # porsi data in-distribution yang akan ditolak
    return tau, is_unk, int(is_unk.sum()), oof_reject, int(trucuk_mask.sum()), needed

runs = {
    "sub_014": ("Anchor Supervised @ standard", trucuk_mask_sup, TARGET_STANDARD_COUNT),
    "sub_015": ("Self-Trained @ standard", trucuk_mask_st, TARGET_STANDARD_COUNT),
    "sub_016": ("Self-Trained @ probe", trucuk_mask_st, TARGET_PROBE_COUNT),
}
res = {}
print("unknown_quantiles_locked:")
for key, (desc, tmask, tgt) in runs.items():
    tau, is_unk, cnt, oof_rej, n_truc, n_dist = calibrate_tau(test_d_perch, tmask, tgt)
    assert cnt == tgt, f"{key}: jumlah Unknown {cnt} != target {tgt}"
    res[key] = dict(tau=tau, is_unk=is_unk, cnt=cnt, oof_rej=oof_rej, n_truc=n_truc, n_dist=n_dist)
    print(f"  {key} ({desc:<28}) tau>={tau:.4f} | Unknown={cnt}/{n_te} ({cnt / n_te:.2%}) | "
          f"dari Trucuk={n_truc}, dari jarak={n_dist} | OOF in-dist yang akan ditolak={oof_rej:.1%}")

# 3. Diagnostik irisan
u14, u15, u16 = (res[k]["is_unk"] for k in ("sub_014", "sub_015", "sub_016"))
ov_14_15 = int((u14 & u15).sum())
assert bool((u16 & ~u15).sum() == 0), "sub_016 seharusnya subset sub_015 (mask Trucuk sama, ambang lebih longgar)"
print("\nunknown_overlap_diagnostics:")
print(f"  sub_014 vs sub_015 mutual Unknown: {ov_14_15}/{res['sub_015']['cnt']} ({ov_14_15 / res['sub_015']['cnt']:.1%})")
print("  sub_016 subset sub_015: terpenuhi secara desain (bukan temuan empiris)")

# 4. Simpan DULU
for key in res:
    np.save(WORK_DIR / f"is_unk_{key.replace('_', '')}.npy", res[key]["is_unk"])

meta_c15 = {
    "rate_standard": RATE_STANDARD,
    "rate_probe": RATE_PROBE,
    "overlap_sub14_sub15": ov_14_15,
    "note": "laju Unknown dipilih dari umpan balik submission (bukan validasi OOF); mask Trucuk memaksa Unknown",
    **{k: {"tau_inclusive": v["tau"], "unknown_count": v["cnt"], "rate": v["cnt"] / n_te,
           "from_trucuk": v["n_truc"], "from_distance": v["n_dist"],
           "oof_indist_reject_rate": v["oof_rej"]} for k, v in res.items()},
}
with open(WORK_DIR / "quantile_locking_meta.json", "w") as fh:
    json.dump(meta_c15, fh, indent=2)

for k, v in res.items():
    if v["n_truc"] > 0.15 * v["cnt"]:
        print(f"[warn] {k}: {v['n_truc']} dari {v['cnt']} Unknown berasal dari flag Trucuk. Cek flag_true_kut di Cell 13.")

print("artifacts_saved: is_unk_sub014.npy, is_unk_sub015.npy, is_unk_sub016.npy, quantile_locking_meta.json")
print(f"cell15_completed | elapsed={time.time() - t0_c15:.2f}s")

unknown_quantiles_locked:
  sub_014 (Anchor Supervised @ standard) tau>=0.5846 | Unknown=354/814 (43.49%) | dari Trucuk=56, dari jarak=298 | OOF in-dist yang akan ditolak=5.7%
  sub_015 (Self-Trained @ standard     ) tau>=0.5833 | Unknown=354/814 (43.49%) | dari Trucuk=52, dari jarak=302 | OOF in-dist yang akan ditolak=5.7%
  sub_016 (Self-Trained @ probe        ) tau>=0.5901 | Unknown=338/814 (41.52%) | dari Trucuk=52, dari jarak=286 | OOF in-dist yang akan ditolak=5.6%

unknown_overlap_diagnostics:
  sub_014 vs sub_015 mutual Unknown: 350/354 (98.9%)
  sub_016 subset sub_015: terpenuhi secara desain (bukan temuan empiris)
[warn] sub_014: 56 dari 354 Unknown berasal dari flag Trucuk. Cek flag_true_kut di Cell 13.
[warn] sub_016: 52 dari 338 Unknown berasal dari flag Trucuk. Cek flag_true_kut di Cell 13.
artifacts_saved: is_unk_sub014.npy, is_unk_sub015.npy, is_unk_sub016.npy, quantile_locking_meta.json
cell15_completed | elapsed=0.01s


In [16]:
# Cell 16 - Assembly, Verification, & Export Tri-Submission (sub_014, sub_015, sub_016)
import base64
import hashlib
import html
import json
import os
import time
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import FileLink, HTML, display

t0_c16 = time.time()
VALID_CLASSES = ["Bondol Jawa", "Cendrawasih", "Kutilang", "Unknown"]
CLASS_TO_IDX = {"Bondol Jawa": 0, "Cendrawasih": 1, "Kutilang": 2}
IDX_TO_CLASS = {v: k for k, v in CLASS_TO_IDX.items()}
CLASS_ARRAY = np.array([IDX_TO_CLASS[i] for i in range(len(IDX_TO_CLASS))], dtype=object)

# 1. Muat Matriks Probabilitas Terkoreksi & Mask Unknown
p_sup_anchor = np.load(WORK_DIR / "test_p_supervised_anchor.npy")     # (814, 3) Cell 9
p_st_adapted = np.load(WORK_DIR / "test_p_selftrained_adapted.npy")   # (814, 3) Cell 13

# FIX: paksa mask jadi boolean (kalau tersimpan sebagai int 0/1, indexing akan salah total)
is_unk_14 = np.load(WORK_DIR / "is_unk_sub014.npy").astype(bool)  # (814,) @ 43.49%
is_unk_15 = np.load(WORK_DIR / "is_unk_sub015.npy").astype(bool)  # (814,) @ 43.49%
is_unk_16 = np.load(WORK_DIR / "is_unk_sub016.npy").astype(bool)  # (814,) @ 41.52%

n_te = len(sub_df)
test_fids = sub_df["file_id"].astype(str).values

# FIX: cek shape & NaN di awal supaya error-nya jelas, bukan error aneh di tengah jalan
for name, arr in [("p_sup_anchor", p_sup_anchor), ("p_st_adapted", p_st_adapted)]:
    assert arr.shape == (n_te, 3), f"{name} shape {arr.shape} != ({n_te}, 3)"
    assert not np.isnan(arr).any(), f"{name} mengandung NaN"
for name, m in [("is_unk_14", is_unk_14), ("is_unk_15", is_unk_15), ("is_unk_16", is_unk_16)]:
    assert m.shape == (n_te,), f"{name} shape {m.shape} != ({n_te},)"

# 2. Rakit Label Prediksi untuk Ketiga Submission
def build_labels(prob_matrix, unk_mask):
    pred_idx = np.argmax(prob_matrix, axis=1)
    labels = CLASS_ARRAY[pred_idx].copy()
    labels[unk_mask] = "Unknown"
    return labels

labels_14 = build_labels(p_sup_anchor, is_unk_14)  # sub_014 Safe Anchor (Supervised @ 43.5%)
labels_15 = build_labels(p_st_adapted, is_unk_15)  # sub_015 SOTA Champion (Self-Trained @ 43.5%)
labels_16 = build_labels(p_st_adapted, is_unk_16)  # sub_016 Conservative Probe (Self-Trained @ 41.5%)

# 3. Verifikasi Integritas Kualitas Ekspor
def validate_submission(labels, name):
    df = pd.DataFrame({"file_id": test_fids, "label": labels})
    assert len(df) == n_te, f"Jumlah baris {name} ({len(df)}) != {n_te}"
    assert df["file_id"].nunique() == n_te, f"file_id pada {name} tidak unik"
    assert df.isna().sum().sum() == 0, f"Ditemukan NaN pada {name}"
    invalid_labels = set(df["label"].unique()) - set(VALID_CLASSES)
    assert not invalid_labels, f"Label ilegal ditemukan di {name}: {invalid_labels}"
    for c in VALID_CLASSES:
        assert (df["label"] == c).sum() > 0, f"Kelas '{c}' memiliki 0 prediksi pada {name}"
    return df

df_14 = validate_submission(labels_14, "sub_014")
df_15 = validate_submission(labels_15, "sub_015")
df_16 = validate_submission(labels_16, "sub_016")

def md5_file(filepath: Path) -> str:
    return hashlib.md5(filepath.read_bytes()).hexdigest()

# 4. Ekspor Berkas Resmi
sub_configs = [
    ("experiment 14.csv", "sub_014_supervised_anchor_p435", df_14, labels_14,
     "Pure_Supervised_Hybrid_1543d_plus_BirdNET_Forensic_at_43.5pct_Unknown",
     "Safe anchor without transductive adaptation; primary safety net for Private LB"),
    ("experiment 15.csv", "sub_015_selftrained_champion_p435", df_15, labels_15,
     "InFold_SelfTrained_Hybrid_1543d_plus_BirdNET_Forensic_at_43.5pct_Unknown",
     "SOTA Champion model with 99 test inliers bridging acoustic domain shift; peak target 0.70+"),
    ("experiment 16.csv", "sub_016_selftrained_probe_p415", df_16, labels_16,
     "InFold_SelfTrained_Hybrid_1543d_plus_BirdNET_Forensic_at_41.5pct_Unknown",
     "Conservative probe testing sensitivity if Private LB has lower Unknown prevalence"),
]

exported_paths = []
print("exporting_tri_submissions:")
for csv_name, sub_id, df, lbls, strat, role in sub_configs:
    sub_dir = WORK_DIR / "submissions" / sub_id
    sub_dir.mkdir(parents=True, exist_ok=True)

    csv_path_top = WORK_DIR / csv_name
    df.to_csv(csv_path_top, index=False)
    df.to_csv(sub_dir / csv_name, index=False)
    exported_paths.append(csv_path_top)

    dist = {k: int(v) for k, v in df["label"].value_counts().items()}
    md5_val = md5_file(csv_path_top)  # FIX: hitung sekali saja
    meta = {
        "submission_id": sub_id,
        "filename": csv_name,
        "strategy": strat,
        "scientific_role": role,
        "n_samples": n_te,
        "unknown_percentage": round(float(np.mean(lbls == "Unknown") * 100), 2),
        "distribution": dist,
        "md5": md5_val,
    }
    with open(sub_dir / "metrics.json", "w") as fh:
        json.dump(meta, fh, indent=2)

    print(f"  {csv_name:<18} -> {dist} | Unk={meta['unknown_percentage']:.2f}% | MD5={md5_val[:8]}")

# FIX: baca ulang file hasil ekspor & pastikan isinya identik dengan df di memori
for (csv_name, _, df, _, _, _), p in zip(sub_configs, exported_paths):
    chk = pd.read_csv(p, dtype=str, keep_default_na=False)
    assert list(chk.columns) == ["file_id", "label"], f"Header {csv_name} salah: {list(chk.columns)}"
    assert (chk["file_id"].values == df["file_id"].values).all(), f"file_id {csv_name} berubah setelah ekspor"
    assert (chk["label"].values == df["label"].values).all(), f"label {csv_name} berubah setelah ekspor"
print("post_export_readback_check = OK")

# Default Kaggle Submission diarahkan ke experiment 15.csv (SOTA Champion)
df_15.to_csv(WORK_DIR / "submission.csv", index=False)
print(f"default_kaggle_submission = experiment 15.csv -> {WORK_DIR / 'submission.csv'}")

# 5. Tabel Komparasi Kesepakatan Antar Submission
agree_14_15 = float(np.mean(labels_14 == labels_15)) * 100.0
agree_15_16 = float(np.mean(labels_15 == labels_16)) * 100.0
agree_14_16 = float(np.mean(labels_14 == labels_16)) * 100.0

print(f"\nsubmission_cross_agreement_matrix:")
print(f"  sub_014 (Anchor) vs sub_015 (Champion SOTA) : {agree_14_15:.2f}% ({int(np.sum(labels_14 != labels_15))} diffs)")
print(f"  sub_015 (Champion) vs sub_016 (Probe 41.5%) : {agree_15_16:.2f}% ({int(np.sum(labels_15 != labels_16))} diffs)")
print(f"  sub_014 (Anchor) vs sub_016 (Probe 41.5%)   : {agree_14_16:.2f}% ({int(np.sum(labels_14 != labels_16))} diffs)")

print("\nCrosstab sub_014 (Anchor, baris) vs sub_015 (Champion, kolom):")
print(pd.crosstab(pd.Series(labels_14, name="sub_014"), pd.Series(labels_15, name="sub_015")))

# 6. Generator Downloader HTML Interaktif Mandiri
def build_html_downloader(items):
    rows = []
    for title, path, df_item in items:  # FIX: jangan pakai nama `p`/`df` biar tidak menimpa variabel global
        b64 = base64.b64encode(path.read_bytes()).decode("ascii")
        dist_str = html.escape(", ".join(f"{k}: {v}" for k, v in df_item["label"].value_counts().items()))
        unk_pct = float(np.mean(df_item["label"] == "Unknown") * 100)
        rows.append(f"""
        <tr>
          <td><b>{html.escape(title)}</b></td>
          <td><a class="btn" download="{html.escape(path.name)}" href="data:text/csv;base64,{b64}">&#11015; {html.escape(path.name)}</a></td>
          <td>{len(df_item)}</td>
          <td><b>{unk_pct:.2f}%</b></td>
          <td class="small">{dist_str}</td>
          <td class="small mono">{md5_file(path)}</td>
        </tr>""")
    return f"""<!DOCTYPE html>
<html lang="id"><head><meta charset="utf-8"><title>Download Day 4 Submissions</title>
<style>
  :root {{ --bg:#ffffff; --fg:#1a1a1a; --line:#ddd; --accent:#1f6feb; }}
  @media (prefers-color-scheme: dark) {{ :root {{ --bg:#1b1b1f; --fg:#eaeaea; --line:#3a3a40; --accent:#58a6ff; }} }}
  body {{ font-family: system-ui, sans-serif; background:var(--bg); color:var(--fg); margin:20px; }}
  table {{ border-collapse:collapse; width:100%; max-width:1150px; }}
  th,td {{ border:1px solid var(--line); padding:8px 12px; text-align:left; }}
  .btn {{ color:var(--accent); font-weight:700; text-decoration:none; }}
  .small {{ font-size:12px; }} .mono {{ font-family:monospace; }}
</style></head><body>
<h2>Download Day 4 Official Submissions (Schematics NPC DSC 2026)</h2>
<p class="small">Klik nama berkas untuk mengunduh langsung dari browser:</p>
<table>
  <tr><th>Eksperimen</th><th>Unduh File</th><th>Baris</th><th>Unknown</th><th>Distribusi Kelas</th><th>MD5 Hash</th></tr>
  {''.join(rows)}
</table></body></html>"""

html_items = [
    ("experiment 14.csv (sub_014 Safe Anchor @ 43.5%)", exported_paths[0], df_14),
    ("experiment 15.csv (sub_015 SOTA Champion @ 43.5%)", exported_paths[1], df_15),
    ("experiment 16.csv (sub_016 Conservative Probe @ 41.5%)", exported_paths[2], df_16),
]

html_doc = build_html_downloader(html_items)
html_path = WORK_DIR / "download_day4_submissions.html"
html_path.write_text(html_doc, encoding="utf-8")

display(HTML(html_doc))

# FIX: fallback FileLink sebelumnya pakai p.name (path salah kalau WORK_DIR != cwd)
cwd = Path.cwd().resolve()
for path in exported_paths:
    rp = path.resolve()
    try:
        link_path = str(rp.relative_to(cwd))
    except ValueError:
        link_path = str(rp)
    display(FileLink(link_path))

print(f"\ncell16_completed | all_3_submissions_exported_successfully | elapsed={time.time()-t0_c16:.2f}s")

exporting_tri_submissions:
  experiment 14.csv  -> {'Unknown': 354, 'Bondol Jawa': 227, 'Kutilang': 150, 'Cendrawasih': 83} | Unk=43.49% | MD5=66ff937e
  experiment 15.csv  -> {'Unknown': 354, 'Bondol Jawa': 245, 'Kutilang': 145, 'Cendrawasih': 70} | Unk=43.49% | MD5=45727adc
  experiment 16.csv  -> {'Unknown': 338, 'Bondol Jawa': 254, 'Kutilang': 150, 'Cendrawasih': 72} | Unk=41.52% | MD5=1ad8d911
post_export_readback_check = OK
default_kaggle_submission = experiment 15.csv -> /kaggle/working/submission.csv

submission_cross_agreement_matrix:
  sub_014 (Anchor) vs sub_015 (Champion SOTA) : 96.31% (30 diffs)
  sub_015 (Champion) vs sub_016 (Probe 41.5%) : 98.03% (16 diffs)
  sub_014 (Anchor) vs sub_016 (Probe 41.5%)   : 94.59% (44 diffs)

Crosstab sub_014 (Anchor, baris) vs sub_015 (Champion, kolom):
sub_015      Bondol Jawa  Cendrawasih  Kutilang  Unknown
sub_014                                                 
Bondol Jawa          223            1         0        3
Cendrawasih      

Eksperimen,Unduh File,Baris,Unknown,Distribusi Kelas,MD5 Hash
experiment 14.csv (sub_014 Safe Anchor @ 43.5%),⬇ experiment 14.csv,814,43.49%,"Unknown: 354, Bondol Jawa: 227, Kutilang: 150, Cendrawasih: 83",66ff937ea59dbfe43202983928f3cdb4
experiment 15.csv (sub_015 SOTA Champion @ 43.5%),⬇ experiment 15.csv,814,43.49%,"Unknown: 354, Bondol Jawa: 245, Kutilang: 145, Cendrawasih: 70",45727adc7c7f83b72b0154d09f9644c3
experiment 16.csv (sub_016 Conservative Probe @ 41.5%),⬇ experiment 16.csv,814,41.52%,"Unknown: 338, Bondol Jawa: 254, Kutilang: 150, Cendrawasih: 72",1ad8d9112b42470bb7159d08a83cabb6


/kaggle/working/experiment 14.csv

/kaggle/working/experiment 15.csv

/kaggle/working/experiment 16.csv


cell16_completed | all_3_submissions_exported_successfully | elapsed=0.07s


In [17]:
# Cell 18 - Intrinsic OOD Rejection (Class-Conditional Spheres & Pure Global OOF Threshold)
import hashlib
import json
import time
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import FileLink, display

t0_c18 = time.time()
VALID_CLASSES = ["Bondol Jawa", "Cendrawasih", "Kutilang", "Unknown"]
CLASS_TO_IDX = {"Bondol Jawa": 0, "Cendrawasih": 1, "Kutilang": 2}
IDX_TO_CLASS = {v: k for k, v in CLASS_TO_IDX.items()}
CLASS_ARRAY = np.array([IDX_TO_CLASS[i] for i in range(len(IDX_TO_CLASS))], dtype=object)

# 1. Muat Fitur Jarak Manifold, Probabilitas Teradaptasi, & Mask Trucuk
oof_d_perch = np.load(WORK_DIR / "oof_d_perch.npy")                  # (1826,)
test_d_perch = np.load(WORK_DIR / "test_d_perch.npy")                # (814,)
p_st_adapted = np.load(WORK_DIR / "test_p_selftrained_adapted.npy")  # (814, 3) Cell 13
trucuk_mask_st = np.load(WORK_DIR / "trucuk_mask_selftrained.npy").astype(bool)

n_te = len(sub_df)
test_fids = sub_df["file_id"].astype(str).values

# FIX: target bisa berupa string ("Bondol Jawa", ...) atau int. Versi lama (y_target == c dengan c int)
# akan menghasilkan mask kosong kalau target berupa string -> np.percentile error / NaN.
y_raw = train_df["target"].values
if y_raw.dtype.kind in "OUS":
    y_target = np.array([CLASS_TO_IDX.get(v, -1) for v in y_raw], dtype=int)
else:
    y_target = y_raw.astype(int)

# FIX: cek shape & NaN di awal (NaN pada jarak bikin perbandingan > selalu False -> lolos sebagai inlier diam-diam)
assert len(y_target) == len(oof_d_perch), f"train target ({len(y_target)}) != oof_d_perch ({len(oof_d_perch)})"
assert test_d_perch.shape == (n_te,), f"test_d_perch shape {test_d_perch.shape} != ({n_te},)"
assert p_st_adapted.shape == (n_te, 3), f"p_st_adapted shape {p_st_adapted.shape} != ({n_te}, 3)"
assert trucuk_mask_st.shape == (n_te,), f"trucuk_mask_st shape {trucuk_mask_st.shape} != ({n_te},)"
assert not np.isnan(oof_d_perch).any(), "oof_d_perch mengandung NaN"
assert not np.isnan(test_d_perch).any(), "test_d_perch mengandung NaN"
assert not np.isnan(p_st_adapted).any(), "p_st_adapted mengandung NaN"

pred_idx_st = np.argmax(p_st_adapted, axis=1)

# 2. Pendekatan 1: sub_017 - Class-Conditional Manifold Spheres (p95 per kelas)
tau_per_class = {}
for c in range(3):
    m = (y_target == c)
    assert m.sum() > 0, f"Kelas {IDX_TO_CLASS[c]} tidak punya sampel OOF (cek format kolom target)"
    tau_per_class[c] = float(np.percentile(oof_d_perch[m], 95.0))

print("class_conditional_thresholds (OOF p95 per class):")
for c in range(3):
    print(f"  {IDX_TO_CLASS[c]:<12}: tau_95 = {tau_per_class[c]:.4f}  (n_oof={int((y_target == c).sum())})")

# FIX: vektorisasi (sebelumnya loop python 814 iterasi, hasilnya sama)
tau_vec = np.array([tau_per_class[c] for c in range(3)])
is_unk_17 = (test_d_perch > tau_vec[pred_idx_st]) | trucuk_mask_st

labels_17 = CLASS_ARRAY[pred_idx_st].copy()
labels_17[is_unk_17] = "Unknown"

# 3. Pendekatan 2: sub_018 - Pure Intrinsic Global OOF Threshold (95% Inlier Retention)
tau_global_95 = float(np.percentile(oof_d_perch, 95.0))
print(f"\npure_global_oof_threshold: tau_95 = {tau_global_95:.4f}")

is_unk_18 = (test_d_perch > tau_global_95) | trucuk_mask_st

labels_18 = CLASS_ARRAY[pred_idx_st].copy()
labels_18[is_unk_18] = "Unknown"

# 4. Validasi dan Ekspor Berkas
def validate_sub(labels, name):
    df = pd.DataFrame({"file_id": test_fids, "label": labels})
    assert len(df) == n_te, f"Jumlah baris {name} ({len(df)}) != {n_te}"
    assert df["file_id"].nunique() == n_te, f"file_id pada {name} tidak unik"
    assert df.isna().sum().sum() == 0, f"Ditemukan NaN pada {name}"
    bad = set(df["label"].unique()) - set(VALID_CLASSES)
    assert not bad, f"Label ilegal di {name}: {bad}"
    for c in VALID_CLASSES:
        assert (df["label"] == c).sum() > 0, f"Kelas '{c}' memiliki 0 prediksi pada {name}"
    return df

df_17 = validate_sub(labels_17, "sub_017")
df_18 = validate_sub(labels_18, "sub_018")

def md5_file(filepath: Path) -> str:
    return hashlib.md5(filepath.read_bytes()).hexdigest()

out_configs = [
    ("experiment 17.csv", "sub_017_class_conditional_spheres", df_17, labels_17,
     "SelfTrained_1543d_Class_Conditional_Manifold_Spheres_p95",
     "Rejection threshold calibrated to each species intrinsic manifold radius on OOF; zero test ranking"),
    ("experiment 18.csv", "sub_018_pure_global_oof_p95", df_18, labels_18,
     "SelfTrained_1543d_Pure_Global_OOF_Inlier_Retention_p95",
     "Rejection threshold calibrated strictly on global OOF p95 retention; zero test-ranking bias"),
]

exported_files = []
print("\nexporting_intrinsic_submissions:")
for csv_name, sub_id, df_item, lbls, strat, role in out_configs:
    sub_dir = WORK_DIR / "submissions" / sub_id
    sub_dir.mkdir(parents=True, exist_ok=True)
    p_top = WORK_DIR / csv_name
    df_item.to_csv(p_top, index=False)
    df_item.to_csv(sub_dir / csv_name, index=False)
    exported_files.append(p_top)

    dist = {k: int(v) for k, v in df_item["label"].value_counts().items()}
    md5_val = md5_file(p_top)
    meta = {
        "submission_id": sub_id,
        "filename": csv_name,
        "strategy": strat,
        "scientific_role": role,
        "n_samples": n_te,
        "unknown_percentage": round(float(np.mean(lbls == "Unknown") * 100), 2),
        "distribution": dist,
        "md5": md5_val,
    }
    with open(sub_dir / "metrics.json", "w") as fh:
        json.dump(meta, fh, indent=2)
    print(f"  {csv_name:<18} -> {dist} | Unk={meta['unknown_percentage']:.2f}% | MD5={md5_val[:8]}")

# FIX: baca ulang hasil ekspor & pastikan identik dengan df di memori
for (csv_name, _, df_item, _, _, _), p_out in zip(out_configs, exported_files):
    chk = pd.read_csv(p_out, dtype=str, keep_default_na=False)
    assert list(chk.columns) == ["file_id", "label"], f"Header {csv_name} salah: {list(chk.columns)}"
    assert (chk["file_id"].values == df_item["file_id"].values).all(), f"file_id {csv_name} berubah setelah ekspor"
    assert (chk["label"].values == df_item["label"].values).all(), f"label {csv_name} berubah setelah ekspor"
print("post_export_readback_check = OK")

# 5. Analisis Komparasi vs sub_015
# FIX: baca sebagai string & pastikan urutan file_id sama dengan test_fids sebelum membandingkan label
# (sebelumnya langsung membandingkan array tanpa cek urutan; file_id juga bisa terbaca sebagai int oleh pandas)
sub15 = pd.read_csv(WORK_DIR / "experiment 15.csv", dtype=str, keep_default_na=False)
assert len(sub15) == n_te and (sub15["file_id"].values == test_fids).all(), \
    "Urutan/isi file_id experiment 15.csv tidak sama dengan sub_df"
labels_15 = sub15["label"].values

agree_15_17 = float(np.mean(labels_15 == labels_17)) * 100.0
agree_15_18 = float(np.mean(labels_15 == labels_18)) * 100.0
agree_17_18 = float(np.mean(labels_17 == labels_18)) * 100.0

print(f"\ncross_agreement_with_sub_015:")
print(f"  sub_015 (Ranked 43.5%) vs sub_017 (Class-Spheres) : {agree_15_17:.2f}% ({int(np.sum(labels_15 != labels_17))} diffs)")
print(f"  sub_015 (Ranked 43.5%) vs sub_018 (Global OOF p95) : {agree_15_18:.2f}% ({int(np.sum(labels_15 != labels_18))} diffs)")
print(f"  sub_017 (Class-Spheres) vs sub_018 (Global OOF p95) : {agree_17_18:.2f}% ({int(np.sum(labels_17 != labels_18))} diffs)")

cwd = Path.cwd().resolve()
for p_out in exported_files:
    try:
        display(FileLink(str(p_out.resolve().relative_to(cwd))))
    except ValueError:
        display(FileLink(str(p_out.resolve())))

print(f"\ncell18_completed | elapsed={time.time()-t0_c18:.2f}s")

class_conditional_thresholds (OOF p95 per class):
  Bondol Jawa : tau_95 = 0.6352  (n_oof=1124)
  Cendrawasih : tau_95 = 0.6611  (n_oof=83)
  Kutilang    : tau_95 = 0.4589  (n_oof=619)

pure_global_oof_threshold: tau_95 = 0.6106

exporting_intrinsic_submissions:
  experiment 17.csv  -> {'Bondol Jawa': 315, 'Unknown': 313, 'Kutilang': 94, 'Cendrawasih': 92} | Unk=38.45% | MD5=d5a39382
  experiment 18.csv  -> {'Unknown': 291, 'Bondol Jawa': 290, 'Kutilang': 155, 'Cendrawasih': 78} | Unk=35.75% | MD5=f33dd4df
post_export_readback_check = OK

cross_agreement_with_sub_015:
  sub_015 (Ranked 43.5%) vs sub_017 (Class-Spheres) : 82.43% (143 diffs)
  sub_015 (Ranked 43.5%) vs sub_018 (Global OOF p95) : 92.26% (63 diffs)
  sub_017 (Class-Spheres) vs sub_018 (Global OOF p95) : 87.71% (100 diffs)


/kaggle/working/experiment 17.csv

/kaggle/working/experiment 18.csv


cell18_completed | elapsed=0.04s


In [18]:
# Cell 19 - Final Archive Zipper: Compress All Working Directory Artifacts & Direct Download
import time
import zipfile
from pathlib import Path
from IPython.display import FileLink, display

t0_zip = time.time()
WORK_PATH = Path("/kaggle/working")
ZIP_NAME = "schematics_day4_all_artifacts.zip"
zip_dest = WORK_PATH / ZIP_NAME

if zip_dest.exists():
    zip_dest.unlink()

print(f"zipping_all_day4_artifacts_in_{WORK_PATH}...")

TARGET_EXTENSIONS = {".csv", ".npy", ".json", ".png", ".txt", ".md", ".html", ".pt"}
file_count = 0
total_raw_bytes = 0

with zipfile.ZipFile(zip_dest, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=6) as zipf:
    for p in sorted(WORK_PATH.rglob("*")):
        if p.is_file() and p.name != ZIP_NAME and not p.name.endswith(".tmp.npy"):
            # Lewati cache chunk mentah BirdNET agar ukuran zip tetap ringkas
            if "birdnet_ckpt" in str(p):
                continue
            if p.suffix.lower() in TARGET_EXTENSIONS:
                arcname = p.relative_to(WORK_PATH)
                zipf.write(p, arcname=arcname)
                file_count += 1
                total_raw_bytes += p.stat().st_size

zip_size_mb = zip_dest.stat().st_size / 1e6
raw_size_mb = total_raw_bytes / 1e6

print("\n" + "=" * 55)
print("DAY 4 ARTIFACTS COMPRESSION COMPLETED!")
print(f"  Total Files Compressed : {file_count} files")
print(f"  Raw Size               : {raw_size_mb:.2f} MB")
print(f"  Zip Archive Size       : {zip_size_mb:.2f} MB (Compression: {total_raw_bytes / max(zip_dest.stat().st_size, 1):.1f}x)")
print(f"  Elapsed Time           : {time.time() - t0_zip:.1f}s")
print("=" * 55)

cwd = Path.cwd().resolve()
try:
    link_path = str(zip_dest.resolve().relative_to(cwd))
except ValueError:
    link_path = str(zip_dest.resolve())

print("KLIK TAUTAN DI BAWAH INI UNTUK MENGUNDUH ARSIP:")
display(FileLink(link_path))
print("=" * 55)

zipping_all_day4_artifacts_in_/kaggle/working...

DAY 4 ARTIFACTS COMPRESSION COMPLETED!
  Total Files Compressed : 70 files
  Raw Size               : 124.62 MB
  Zip Archive Size       : 114.45 MB (Compression: 1.1x)
  Elapsed Time           : 5.7s
KLIK TAUTAN DI BAWAH INI UNTUK MENGUNDUH ARSIP:


/kaggle/working/schematics_day4_all_artifacts.zip